In [1]:
from pathlib import Path
from datetime import datetime, timezone
from copy import deepcopy
import hashlib
import re
import json
import joblib
import numpy as np
import pandas as pd
import optuna

from scipy.sparse import csr_matrix, hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder, StandardScaler, normalize

DATA_PATH = Path("nepal_tourism_data.csv")
ARTIFACT_DIR = Path("tourism_recommender_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

ARTIFACT_VERSION = "2.0.0"
SCHEMA_VERSION = "content_based_recommender.v2"
MODEL_RANDOM_STATE = 42
PRODUCTION_VECTOR_DIM = 128

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)


In [2]:
df_raw = pd.read_csv(DATA_PATH)

print("Raw dataset shape:", df_raw.shape)


Raw dataset shape: (1440, 53)


In [3]:
df_raw.head(3)


,title,subtitle,description,overview,highlights,itinerary,inclusions,exclusions,good_to_know,review_snippet,destination,location,start_location,end_location,route_summary,destinations_visited,category,activity_type,style,theme,traveler_type,difficulty_level,ideal_for,tags,price_display,price_amount,price_currency,duration_text,duration_type,duration_days,duration_nights,average_duration_hours,best_seasons,cancellation_policy,languages_offered,group_type,group_size_note,max_group_size,min_group_size,guide_language,transport,accommodation,meals,age_range,availability,instant_confirmation,brochure_available,rating_average,rating_count,booking_count,special_offer,is_bookable,is_active
0,Kathmandu UNESCO Heritage Circuit with Private Guide,Private landmark circuit across Kathmandu’s major heritage stops.,"This full-day Kathmandu circuit is built for travelers who want the major UNESCO-era monuments without turning the day into a blur. The route starts in the old royal square, climbs to Swayambhunat...","Private or small-group heritage day in Kathmandu linking Kathmandu Durbar Square, Swayambhunath, and Pashupatinath with guide support, monument context, and flexible stop times.",Three very different Kathmandu heritage environments in one day | Easy progression from palace square to hilltop stupa to active temple complex | Good choice when you want context without arrangin...,Pickup in Kathmandu | Guided visit at Kathmandu Durbar Square | Continue to Swayambhunath | Explore Pashupatinath | Drop-off in Kathmandu,Guide service | Private or shared transport | Meeting point assistance,Monument entry fees | Personal expenses | Tips,Monument entry tickets are usually paid on site in cash | Expect stair climbing at Swayambhunath and uneven paving in the old square | Pashupatinath is busiest around ritual hours and cremation ac...,Good choice for seeing several heritage spots without logistics stress.,Kathmandu,Kathmandu,Kathmandu,Kathmandu,Kathmandu Durbar Square → Swayambhunath → Pashupatinath,Kathmandu | Kathmandu Durbar Square | Swayambhunath | Pashupatinath,Culture | Landmarks,CITY_TOUR,Small-group guided,"Historic quarters, temples, and lived-in city spaces",First-time visitors | Couples | Solo travelers,EASY,First-time visitors to Kathmandu | Travelers who want a guide for the city’s major heritage sites,Kathmandu | UNESCO heritage day | Durbar Square | Swayambhunath | Pashupatinath,USD 48,48,USD,7 hours,FULL_DAY,NaN,NaN,7.0,YEAR_ROUND,Free cancellation up to 48 hours before start,English | Hindi,Private guided,Private departures cost more but offer flexibility,10,1,English,Private car with short walking sections,NaN,Optional local lunch stop,5+,"Daily, selected times",True,True,4.6,127,478,NaN,True,True
1,"Pashupatinath, Boudhanath and Patan Heritage Day",Private or small-group sightseeing across Kathmandu.,"Kathmandu works best with a route that moves logically between major squares, shrines, and older streets. This day leans into the contrast between the smoke and ceremony of Pashupatinath, the wide...","Guided day trip linking Pashupatinath, Boudhanath, and Patan Durbar Square with transfers arranged and time for explanation, observation, and short breaks.",See Pashupatinath and Boudhanath in one well-paced day | Context from a local guide instead of independent stop-hopping | Good balance of landmark stops and local atmosphere,Pickup in Kathmandu | Guided visit at Pashupatinath | Continue to Boudhanath | Explore Patan Durbar Square | Drop-off in Kathmandu,Guide service | Private or shared transport | Meeting point assistance,Monument entry fees | Personal expenses | Tips,"Carry local cash for monument tickets, tea, and small purchases | Early starts usually mean quieter courtyards and easier parking | Allow extra time if you want souvenir or handicraft stops","A useful mix of history, architecture, and practical local insight.",Kathmandu,Kathmandu,Kathmandu,Kathmandu,Pashupatinath → Boudhanath → Patan Durbar Square,Kathmandu | P

In [4]:
print("Missing ratio (top 20):")
display(df_raw.isna().mean().sort_values(ascending=False).head(20))

print("\nDuplicate title count:", df_raw["title"].duplicated().sum())
print("Unique titles:", df_raw["title"].nunique())


Missing ratio (top 20):


special_offer             0.834028
accommodation             0.567361
duration_days             0.389583
duration_nights           0.389583
average_duration_hours    0.239583
meals                     0.083333
transport                 0.059028
min_group_size            0.000000
best_seasons              0.000000
cancellation_policy       0.000000
languages_offered         0.000000
group_type                0.000000
group_size_note           0.000000
max_group_size            0.000000
title                     0.000000
guide_language            0.000000
duration_text             0.000000
age_range                 0.000000
availability              0.000000
instant_confirmation      0.000000
dtype: float64


Duplicate title count: 0
Unique titles: 1440


# Canonical Preprocessing And Backend Contract

This notebook stays fully independent and is not intended to be shipped into Django. The production integration target is a portable transform bundle that Django can use to preprocess live DB listings into dense vectors for `pgvector` retrieval.

The sparse TF-IDF + metadata recommender remains here for offline evaluation, debugging, and human-readable recommendation inspection only. The production-facing path is the dense `TruncatedSVD` projection plus L2 normalization.

Important identity note: notebook `item_id` values are training and evaluation identifiers only. Django live indexing must use the real DB primary key or UUID as the production item identity.

Important serving note: the exported bundle is for transform only, not direct recommendation serving. Django should use the bundle to build dense vectors, `pgvector` should perform nearest-neighbor retrieval over live DB listing vectors, and the backend should then apply business-rule filtering, caching, reranking, and API formatting.

In [5]:
REQUIRED_RAW_COLUMNS = [
    "title",
    "description",
    "destination",
    "category",
    "activity_type",
    "difficulty_level",
    "duration_type",
    "duration_days",
    "duration_nights",
    "average_duration_hours",
    "duration_text",
    "tags",
    "ideal_for",
    "best_seasons",
    "languages_offered",
    "max_group_size",
    "instant_confirmation",
    "is_bookable",
    "is_active",
]

MULTI_VALUE_SOURCE_COLS = [
    "tags",
    "ideal_for",
    "best_seasons",
    "languages_offered",
]

TEXT_BASE_COLS = [
    "title",
    "description",
    "destination_norm",
    "destination_group",
]

CATEGORICAL_COLS = [
    "category_mapped",
    "activity_type_for_ohe",
    "difficulty_level_norm",
    "duration_type_mapped",
]

MULTI_VALUE_COLS = [f"{col}_list" for col in MULTI_VALUE_SOURCE_COLS]

NUMERIC_SOURCE_COLS = [
    "duration_days",
    "duration_nights",
    "average_duration_hours",
    "max_group_size",
]

NUMERIC_COLS = [
    "duration_days_filled",
    "duration_nights_filled",
    "average_duration_hours_filled",
    "max_group_size",
]

BOOLEAN_COLS = [
    "instant_confirmation",
    "is_bookable",
    "is_active",
]

# These fields remain in the canonical schema but are intentionally excluded
# from semantic similarity.
SEMANTIC_BOOLEAN_COLS = []
NON_SEMANTIC_RUNTIME_FIELDS = [
    "instant_confirmation",
    "is_bookable",
    "is_active",
]

DESTINATION_ALIASES = {
    "kathmandu valley": "kathmandu valley",
    "kathmandu": "kathmandu",
    "pokhara valley": "pokhara",
    "pokhara": "pokhara",
    "sarangkot": "sarangkot",
    "chitwan national park": "chitwan",
    "chitwan": "chitwan",
    "lumbini development zone": "lumbini",
    "lumbini": "lumbini",
    "nagarkot": "nagarkot",
    "bhaktapur": "bhaktapur",
    "patan": "patan",
    "bandipur": "bandipur",
    "mustang": "mustang",
    "annapurna region": "annapurna region",
    "everest region": "everest region",
    "langtang region": "langtang region",
    "bhote koshi": "bhote koshi",
}

DESTINATION_GROUP_MAP = {
    "kathmandu": "kathmandu_valley",
    "kathmandu valley": "kathmandu_valley",
    "patan": "kathmandu_valley",
    "bhaktapur": "kathmandu_valley",
    "nagarkot": "kathmandu_valley",
    "pokhara": "pokhara_annapurna",
    "sarangkot": "pokhara_annapurna",
    "annapurna region": "pokhara_annapurna",
    "chitwan": "chitwan_region",
    "lumbini": "lumbini_region",
    "mustang": "mustang_region",
    "everest region": "everest_region",
    "langtang region": "langtang_region",
    "bandipur": "bandipur_region",
    "bhote koshi": "bhote_koshi_region",
}

ACTIVITY_MAP = {
    "TREKKING": "trekking",
    "CITY_TOUR": "city_tour",
    "HIKING": "hiking",
    "TEMPLE_VISIT": "temple_visit",
    "HOMESTAY": "homestay",
    "COOKING_CLASS": "cooking_class",
    "BIRD_WATCHING": "bird_watching",
    "CYCLING": "cycling",
    "RAFTING": "rafting",
    "MEDITATION": "meditation",
    "WILDLIFE_SAFARI": "wildlife_safari",
    "YOGA": "yoga",
    "MOUNTAIN_FLIGHT": "mountain_flight",
    "PARAGLIDING": "paragliding",
    "BUNGEE": "bungee",
    "OVERLAND_ADVENTURE": "overland_adventure",
    "CULTURAL_WORKSHOP": "cultural_workshop",
    "FOOD_TOUR": "food_tour",
    "BOATING": "boating",
    "ROCK_CLIMBING": "rock_climbing",
}

DIFFICULTY_FALLBACK_MAP = {
    "hard": "difficult",
}

DURATION_TYPE_MAP = {
    "HALF_DAY": "half_day",
    "FULL_DAY": "full_day",
    "MULTI_DAY": "multi_day",
}

EXPECTED_DURATION_TYPE_VALUES = ["half_day", "full_day", "multi_day"]
EXPECTED_DIFFICULTY_VALUES = ["easy", "moderate", "challenging", "difficult", "expert"]

PREFIXED_TOKEN_RECIPE = {
    "destination_repeat": 8,
    "destination_group_repeat": 5,
    "category_repeat": 3,
    "difficulty_repeat": 2,
    "duration_repeat": 2,
    "activity_repeat": 4,
    "tag_repeat": 2,
    "ideal_repeat": 1,
    "season_repeat": 1,
    "language_repeat": 1,
}

CATEGORY_RULE_SEQUENCE = [
    "festival",
    "trekking",
    "wellness",
    "food_tours",
    "village",
    "nature",
    "photography",
    "heritage",
    "adventure",
]

CATEGORY_ACTIVITY_RULES = {
    "festival": [],
    "trekking": ["TREKKING"],
    "wellness": ["YOGA", "MEDITATION"],
    "food_tours": ["COOKING_CLASS", "FOOD_TOUR"],
    "village": ["HOMESTAY"],
    "nature": ["BIRD_WATCHING", "WILDLIFE_SAFARI"],
    "photography": [],
    "heritage": ["TEMPLE_VISIT"],
    "adventure": [
        "HIKING",
        "CYCLING",
        "RAFTING",
        "PARAGLIDING",
        "BUNGEE",
        "MOUNTAIN_FLIGHT",
        "OVERLAND_ADVENTURE",
        "BOATING",
        "ROCK_CLIMBING",
    ],
}

CATEGORY_KEYWORD_RULES = {
    "festival": ["festival"],
    "trekking": ["trekking", "teahouse", "base camp", "himalaya treks"],
    "wellness": ["wellness", "retreat", "mindfulness", "yoga", "meditation"],
    "food_tours": ["food", "kitchen", "cooking", "culinary"],
    "village": ["homestay", "community", "village", "rural", "local life"],
    "nature": ["wildlife", "birding", "jungle", "forest", "nature"],
    "photography": ["photography", "photo"],
    "heritage": ["heritage", "unesco", "old town", "durbar", "pilgrimage", "monaster", "sacred", "temple", "landmark"],
    "adventure": ["adventure", "outdoor", "adrenaline", "flightseeing", "air tours", "scenic flights", "viewpoints", "sunrise trips"],
}

CATEGORY_TEXT_FIELDS_USED = ["category", "title", "description", "tags"]
CATEGORY_FALLBACK = "cultural"

IDENTITY_CONTRACT = {
    "notebook_item_id_role": "Training and evaluation identifier only.",
    "production_identity_requirement": "Django live indexing must use the DB primary key or UUID as the production item identity.",
    "stability_note": "Notebook item_id values can change if content fields change, so they must not be treated as live production IDs.",
}

BACKEND_SERVING_NOTE = (
    "Use the exported bundle only to transform listings into dense vectors. "
    "pgvector should perform nearest-neighbor retrieval over live DB listing vectors. "
    "The notebook artifacts are not the direct recommendation-serving engine that returns final listing IDs."
)

DEFAULT_FILL_BEHAVIOR = {
    "text_fields": "",
    "list_fields": [],
    "boolean_fields": False,
    "destination_norm": "unknown_destination when source destination is blank",
    "activity_type_mapped": "unknown when source activity_type is blank",
    "difficulty_level_norm": "easy when value is blank or out of vocabulary",
    "duration_type_mapped": {
        "HALF_DAY": "half_day",
        "FULL_DAY": "full_day",
        "MULTI_DAY": "multi_day",
        "default": "full_day",
    },
    "duration_days_filled": {
        "half_day": 0.5,
        "full_day": 1.0,
        "multi_day": 2.0,
    },
    "duration_nights_filled": {
        "half_day": 0.0,
        "full_day": 0.0,
        "multi_day": "max(duration_days_filled - 1, 1)",
    },
    "average_duration_hours_filled": {
        "half_day": 4.0,
        "full_day": 7.0,
        "multi_day": "duration_days_filled * 8.0",
    },
    "max_group_size": 0.0,
}

DEFAULT_FEATURE_CONFIG = {
    "weights": {
        "tfidf": 1.0,
        "categorical": 0.8,
        "multi_value": 0.5,
        "numeric": 0.3,
    },
    "rerank_bonus": {
        "destination": 0.22,
        "destination_group": 0.10,
        "activity": 0.12,
        "category": 0.08,
        "difficulty": 0.04,
    },
    "candidate_pool": 20,
    "top_k": 5,
    "tfidf": {
        "max_features": 10000,
        "ngram_range": (1, 2),
        "min_df": 3,
        "max_df": 0.85,
        "sublinear_tf": True,
        "stop_words": "english",
    },
    "dense_projection": {
        "target_dim": PRODUCTION_VECTOR_DIM,
        "normalization": "l2",
        "random_state": MODEL_RANDOM_STATE,
    },
}

CANONICAL_OUTPUT_COLUMNS = [
    "item_id",
    "source_name",
    "source_row_number",
    "title",
    "description",
    "destination",
    "destination_norm",
    "destination_group",
    "category",
    "category_mapped",
    "activity_type",
    "activity_type_mapped",
    "activity_type_for_ohe",
    "difficulty_level",
    "difficulty_level_norm",
    "duration_type",
    "duration_type_mapped",
    "duration_text",
    "duration_days",
    "duration_days_filled",
    "duration_nights",
    "duration_nights_filled",
    "average_duration_hours",
    "average_duration_hours_filled",
    "tags",
    "tags_list",
    "ideal_for",
    "ideal_for_list",
    "best_seasons",
    "best_seasons_list",
    "languages_offered",
    "languages_offered_list",
    "max_group_size",
    "instant_confirmation",
    "is_bookable",
    "is_active",
    "meta_token_text",
    "combined_text",
]

In [6]:
def split_pipe(value):
    if value is None:
        return []
    if isinstance(value, (float, np.floating)) and pd.isna(value):
        return []
    if isinstance(value, (list, tuple, set, np.ndarray)):
        parts = [clean_text(part) for part in value]
        return [part for part in parts if part]
    if isinstance(value, str):
        stripped = value.strip()
        if not stripped:
            return []
        if stripped.startswith("[") and stripped.endswith("]"):
            try:
                parsed = json.loads(stripped)
            except json.JSONDecodeError:
                parsed = None
            if isinstance(parsed, (list, tuple, set)):
                parts = [clean_text(part) for part in parsed]
                return [part for part in parts if part]
    if pd.isna(value):
        return []
    parts = [clean_text(part) for part in str(value).split("|")]
    return [part for part in parts if part]

def clean_text(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()
    value = re.sub(r"\s+", " ", value)
    return value

def slugify(value):
    value = clean_text(value)
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = value.strip("_")
    return value or "item"

def normalize_boolean(value):
    if pd.isna(value):
        return False
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, np.integer, float, np.floating)):
        return bool(value)
    return str(value).strip().lower() in {"1", "true", "t", "yes", "y"}

def normalize_destination(value):
    value = clean_text(value)
    if not value:
        return "unknown_destination"
    return DESTINATION_ALIASES.get(value, value)

def map_activity_type(raw):
    if pd.isna(raw) or str(raw).strip() == "":
        return "unknown"
    raw_key = str(raw).strip().upper()
    return ACTIVITY_MAP.get(raw_key, clean_text(raw_key).replace(" ", "_"))

def normalize_difficulty(value):
    value = clean_text(value)
    if not value:
        return "easy"
    if value in EXPECTED_DIFFICULTY_VALUES:
        return value
    return DIFFICULTY_FALLBACK_MAP.get(value, "easy")

def normalize_duration_type(value):
    raw = "" if pd.isna(value) else str(value).strip().upper()
    return DURATION_TYPE_MAP.get(raw, "full_day")

def parse_duration_text(text):
    if pd.isna(text):
        return {}
    text = str(text).lower()
    parsed = {}
    match = re.search(r"(\d+(?:\.\d+)?)\s*hours?", text)
    if match:
        parsed["hours"] = float(match.group(1))
    match = re.search(r"(\d+(?:\.\d+)?)\s*days?", text)
    if match:
        parsed["days"] = float(match.group(1))
    match = re.search(r"(\d+(?:\.\d+)?)\s*nights?", text)
    if match:
        parsed["nights"] = float(match.group(1))
    return parsed

def map_category(row):
    raw_activity = "" if pd.isna(row.get("activity_type")) else str(row["activity_type"]).strip().upper()
    text_blob = " ".join(clean_text(row.get(field)) for field in CATEGORY_TEXT_FIELDS_USED)

    for category_name in CATEGORY_RULE_SEQUENCE:
        activity_match = raw_activity in CATEGORY_ACTIVITY_RULES.get(category_name, [])
        keyword_match = any(keyword in text_blob for keyword in CATEGORY_KEYWORD_RULES.get(category_name, []))
        if activity_match or keyword_match:
            return category_name

    return CATEGORY_FALLBACK

def build_prefixed_tokens(row):
    tokens = []

    destination = row["destination_norm"].replace(" ", "_") if row["destination_norm"] else ""
    destination_group = row["destination_group"].replace(" ", "_") if row["destination_group"] else ""
    category = row["category_mapped"]
    difficulty = row["difficulty_level_norm"]
    duration = row["duration_type_mapped"]
    activity = row["activity_type_mapped"]

    if destination:
        tokens.extend([f"destination_{destination}"] * PREFIXED_TOKEN_RECIPE["destination_repeat"])
    if destination_group:
        tokens.extend([f"destinationgroup_{destination_group}"] * PREFIXED_TOKEN_RECIPE["destination_group_repeat"])

    tokens.extend([f"category_{category}"] * PREFIXED_TOKEN_RECIPE["category_repeat"])
    tokens.extend([f"difficulty_{difficulty}"] * PREFIXED_TOKEN_RECIPE["difficulty_repeat"])
    tokens.extend([f"duration_{duration}"] * PREFIXED_TOKEN_RECIPE["duration_repeat"])
    tokens.extend([f"activity_{activity}"] * PREFIXED_TOKEN_RECIPE["activity_repeat"])

    for tag in row["tags_list"]:
        tokens.extend([f"tag_{tag.replace(' ', '_')}"] * PREFIXED_TOKEN_RECIPE["tag_repeat"])
    for value in row["ideal_for_list"]:
        tokens.extend([f"ideal_{value.replace(' ', '_')}"] * PREFIXED_TOKEN_RECIPE["ideal_repeat"])
    for value in row["best_seasons_list"]:
        tokens.extend([f"season_{value.replace(' ', '_')}"] * PREFIXED_TOKEN_RECIPE["season_repeat"])
    for value in row["languages_offered_list"]:
        tokens.extend([f"lang_{value.replace(' ', '_')}"] * PREFIXED_TOKEN_RECIPE["language_repeat"])

    return " ".join(tokens)

def build_item_id(row, source_name):
    fingerprint = "||".join(
        [
            source_name,
            clean_text(row.get("title")),
            clean_text(row.get("destination_norm")),
            clean_text(row.get("category_mapped")),
            clean_text(row.get("activity_type_mapped")),
            clean_text(row.get("duration_type_mapped")),
            clean_text(row.get("description"))[:160],
        ]
    )
    digest = hashlib.sha1(fingerprint.encode("utf-8")).hexdigest()[:16]
    title_slug = slugify(row.get("title"))[:40]
    return f"{source_name}_{title_slug}_{digest}"

def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

def make_feature_config(overrides=None):
    feature_config = deepcopy(DEFAULT_FEATURE_CONFIG)
    overrides = overrides or {}
    for key, value in overrides.items():
        if isinstance(value, dict) and isinstance(feature_config.get(key), dict):
            feature_config[key].update(value)
        else:
            feature_config[key] = value
    return feature_config

def make_json_safe(value):
    if isinstance(value, dict):
        return {str(k): make_json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(v) for v in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value

In [7]:
def prepare_items(df_raw, source_name="scraped"):
    df = df_raw.copy()

    for column in REQUIRED_RAW_COLUMNS:
        if column not in df.columns:
            if column in BOOLEAN_COLS:
                df[column] = False
            elif column in NUMERIC_SOURCE_COLS:
                df[column] = np.nan
            else:
                df[column] = ""

    df["source_name"] = source_name
    df["source_row_number"] = np.arange(len(df))

    text_columns = [
        "title",
        "description",
        "destination",
        "category",
        "activity_type",
        "difficulty_level",
        "duration_type",
        "duration_text",
    ]
    for column in text_columns:
        df[column] = df[column].fillna("").astype(str)

    for column in MULTI_VALUE_SOURCE_COLS:
        df[column] = df[column].fillna("").astype(str)
        df[f"{column}_list"] = df[column].apply(split_pipe)

    for column in NUMERIC_SOURCE_COLS:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    for column in BOOLEAN_COLS:
        df[column] = df[column].apply(normalize_boolean)

    df["destination_norm"] = df["destination"].apply(normalize_destination)
    df["destination_group"] = df["destination_norm"].map(
        lambda value: DESTINATION_GROUP_MAP.get(value, value if value else "unknown_destination")
    )

    df["category_mapped"] = df.apply(map_category, axis=1)
    df["activity_type_mapped"] = df["activity_type"].apply(map_activity_type)
    df["activity_type_for_ohe"] = df["activity_type_mapped"].fillna("unknown")
    df["difficulty_level_norm"] = df["difficulty_level"].apply(normalize_difficulty)
    df["duration_type_mapped"] = df["duration_type"].apply(normalize_duration_type)

    parsed_duration = df["duration_text"].apply(parse_duration_text)

    df["duration_days_filled"] = df["duration_days"]
    df["duration_nights_filled"] = df["duration_nights"]
    df["average_duration_hours_filled"] = df["average_duration_hours"]

    for row_position, parsed in parsed_duration.items():
        if pd.isna(df.at[row_position, "average_duration_hours_filled"]) and "hours" in parsed:
            df.at[row_position, "average_duration_hours_filled"] = parsed["hours"]
        if pd.isna(df.at[row_position, "duration_days_filled"]) and "days" in parsed:
            df.at[row_position, "duration_days_filled"] = parsed["days"]
        if pd.isna(df.at[row_position, "duration_nights_filled"]) and "nights" in parsed:
            df.at[row_position, "duration_nights_filled"] = parsed["nights"]

    default_duration_days = np.select(
        [
            df["duration_type_mapped"] == "half_day",
            df["duration_type_mapped"] == "multi_day",
        ],
        [0.5, 2.0],
        default=1.0,
    )

    df["duration_days_filled"] = np.where(
        df["duration_days_filled"].isna(),
        default_duration_days,
        df["duration_days_filled"],
    )

    df["duration_nights_filled"] = np.where(
        df["duration_nights_filled"].isna(),
        np.where(
            df["duration_type_mapped"] == "multi_day",
            np.maximum(df["duration_days_filled"] - 1, 1),
            0.0,
        ),
        df["duration_nights_filled"],
    )

    df["average_duration_hours_filled"] = np.where(
        df["average_duration_hours_filled"].isna(),
        np.select(
            [
                df["duration_type_mapped"] == "half_day",
                df["duration_type_mapped"] == "full_day",
            ],
            [
                4.0,
                7.0,
            ],
            default=df["duration_days_filled"] * 8.0,
        ),
        df["average_duration_hours_filled"],
    )

    df["max_group_size"] = df["max_group_size"].fillna(0.0).astype(float)

    df["meta_token_text"] = df.apply(build_prefixed_tokens, axis=1)
    df["combined_text"] = (
        df["title"].str.lower().fillna("")
        + " "
        + df["title"].str.lower().fillna("")
        + " "
        + df["description"].str.lower().fillna("")
        + " "
        + df["destination_norm"].fillna("")
        + " "
        + df["destination_norm"].fillna("")
        + " "
        + df["destination_group"].fillna("")
        + " "
        + df["destination_group"].fillna("")
        + " "
        + df["meta_token_text"].fillna("")
    ).str.replace(r"\s+", " ", regex=True).str.strip()

    df["item_id_base"] = df.apply(lambda row: build_item_id(row, source_name=source_name), axis=1)
    df["item_id_ordinal"] = df.groupby("item_id_base").cumcount().add(1)
    df["item_id"] = np.where(
        df["item_id_ordinal"].eq(1),
        df["item_id_base"],
        df["item_id_base"] + "_" + df["item_id_ordinal"].astype(str),
    )

    canonical_df = df[CANONICAL_OUTPUT_COLUMNS].copy()
    return canonical_df


In [8]:
train_items = prepare_items(df_raw, source_name="scraped")

print("Canonical training shape:", train_items.shape)
display(
    train_items[
        [
            "item_id",
            "title",
            "destination_norm",
            "destination_group",
            "category_mapped",
            "activity_type_mapped",
            "difficulty_level_norm",
            "duration_type_mapped",
        ]
    ].head(10)
)


Canonical training shape: (1440, 38)


,item_id,title,destination_norm,destination_group,category_mapped,activity_type_mapped,difficulty_level_norm,duration_type_mapped
0,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,Kathmandu UNESCO Heritage Circuit with Private Guide,kathmandu,kathmandu_valley,heritage,city_tour,easy,full_day
1,scraped_pashupatinath_boudhanath_and_patan_herit_c1faa00f8590b3d8,"Pashupatinath, Boudhanath and Patan Heritage Day",kathmandu,kathmandu_valley,heritage,city_tour,easy,full_day
2,scraped_kathmandu_valley_unesco_sites_day_tour_bae70cfbba82963e,Kathmandu Valley UNESCO Sites Day Tour,kathmandu valley,kathmandu_valley,heritage,city_tour,easy,full_day
3,scraped_bhaktapur_durbar_square_and_old_town_wal_28a5a611f448b797,Bhaktapur Durbar Square and Old Town Walk,bhaktapur,kathmandu_valley,heritage,city_tour,easy,half_day
4,scraped_patan_museum_golden_temple_and_courtyard_4256bea42560d249,"Patan Museum, Golden Temple and Courtyard Walk",patan,kathmandu_valley,heritage,city_tour,easy,half_day
5,scraped_thamel_asan_and_indra_chowk_bazaar_walk_f218b7fa14307a36,"Thamel, Asan and Indra Chowk Bazaar Walk",kathmandu,kathmandu_valley,heritage,city_tour,easy,half_day
6,scraped_boudhanath_and_pashupatinath_spiritual_c_f24413cd34e583e8,Boudhanath and Pashupatinath Spiritual Circuit,kathmandu,kathmandu_valley,heritage,temple_visit,easy,half_day
7,scraped_bhaktapur_and_changu_narayan_heritage_da_e9f37a3a88c42b08,Bhaktapur and Changu Narayan Heritage Day Trip,bhaktapur,kathmandu_valley,village,city_tour,easy,half_day
8,scraped_patan_and_kirtipur_cultural_day_out_6439cfb58cb1c30d,Patan and Kirtipur Cultural Day Out,patan,kathmandu_valley,heritage,city_tour,easy,full_day
9,scraped_bhaktapur_and_patan_highlights_from_kath_34eb585502c955e3,Bhaktapur and Patan Highlights from Kathmandu,kathmandu valley,kathmandu_valley,heritage,city_tour,easy,full_day


# Feature Engineering And pgvector Bridge

The notebook still builds the original sparse representation because it is useful for offline KNN evaluation and sanity checks. The production-facing path is the dense `TruncatedSVD` projection of that same sparse matrix, followed by L2 normalization so the resulting vectors can be stored in `pgvector`.


In [9]:
def fit_feature_components(train_items, feature_config):
    vectorizer = TfidfVectorizer(
        max_features=feature_config["tfidf"]["max_features"],
        ngram_range=feature_config["tfidf"]["ngram_range"],
        min_df=feature_config["tfidf"]["min_df"],
        max_df=feature_config["tfidf"]["max_df"],
        stop_words=feature_config["tfidf"]["stop_words"],
        sublinear_tf=feature_config["tfidf"]["sublinear_tf"],
    )

    cat_encoder = make_onehot_encoder()
    multilabel_binarizers = {
        column: MultiLabelBinarizer(sparse_output=True)
        for column in MULTI_VALUE_COLS
    }
    scaler = StandardScaler()

    vectorizer.fit(train_items["combined_text"])
    cat_encoder.fit(train_items[CATEGORICAL_COLS])

    for column in MULTI_VALUE_COLS:
        multilabel_binarizers[column].fit(train_items[column])

    scaler.fit(train_items[NUMERIC_COLS].astype(float))

    return {
        "vectorizer": vectorizer,
        "cat_encoder": cat_encoder,
        "multilabel_binarizers": multilabel_binarizers,
        "scaler": scaler,
    }

def transform_items_to_sparse(canonical_items, feature_components, feature_config):
    weights = feature_config["weights"]
    matrices = []

    X_text = feature_components["vectorizer"].transform(canonical_items["combined_text"])
    matrices.append(X_text.multiply(weights["tfidf"]))

    X_cat = feature_components["cat_encoder"].transform(canonical_items[CATEGORICAL_COLS])
    matrices.append(X_cat.multiply(weights["categorical"]))

    for column in MULTI_VALUE_COLS:
        matrix = feature_components["multilabel_binarizers"][column].transform(canonical_items[column])
        matrices.append(matrix.multiply(weights["multi_value"]))

    X_num_dense = feature_components["scaler"].transform(canonical_items[NUMERIC_COLS].astype(float))
    matrices.append(csr_matrix(X_num_dense).multiply(weights["numeric"]))

    if SEMANTIC_BOOLEAN_COLS:
        X_bool = csr_matrix(canonical_items[SEMANTIC_BOOLEAN_COLS].astype(int).values)
        matrices.append(X_bool)

    return hstack(matrices, format="csr")

def fit_dense_projector(X_sparse_train, feature_config):
    max_components = max(2, min(X_sparse_train.shape[0] - 1, X_sparse_train.shape[1] - 1))
    n_components = min(feature_config["dense_projection"]["target_dim"], max_components)

    dense_projector = TruncatedSVD(
        n_components=n_components,
        random_state=feature_config["dense_projection"]["random_state"],
    )
    X_dense = dense_projector.fit_transform(X_sparse_train)
    X_dense = normalize(
        X_dense,
        norm=feature_config["dense_projection"]["normalization"],
        axis=1,
    )
    return dense_projector, X_dense

def transform_sparse_to_dense(X_sparse, dense_projector, feature_config):
    X_dense = dense_projector.transform(X_sparse)
    X_dense = normalize(
        X_dense,
        norm=feature_config["dense_projection"]["normalization"],
        axis=1,
    )
    return X_dense

def transform_items_to_dense(canonical_items, feature_components, dense_projector, feature_config):
    X_sparse = transform_items_to_sparse(canonical_items, feature_components, feature_config)
    return transform_sparse_to_dense(X_sparse, dense_projector, feature_config)

def transform_single_item_to_dense(raw_item, feature_components, dense_projector, feature_config, source_name="live"):
    if isinstance(raw_item, pd.Series):
        raw_item = raw_item.to_dict()
    prepared_single = prepare_items(pd.DataFrame([raw_item]), source_name=source_name)
    dense_vectors = transform_items_to_dense(
        prepared_single,
        feature_components,
        dense_projector,
        feature_config,
    )
    return prepared_single, dense_vectors[0]

def fit_knn_index(feature_matrix):
    knn = NearestNeighbors(metric="cosine", algorithm="brute")
    knn.fit(feature_matrix)
    return knn


# Offline Debug Recommender vs Production Transform Artifacts

`instant_confirmation`, `is_bookable`, and `is_active` are preserved in the canonical dataframe and exported preprocessing contract, but they are intentionally grouped as `NON_SEMANTIC_RUNTIME_FIELDS` rather than semantic similarity signals.

Notebook `item_id` values are not production identities. They are training and evaluation identifiers for notebook artifacts, while Django live indexing must use the real DB PK or UUID.

In [10]:
def rerank_candidates(seed_row, candidate_pairs, candidate_frame, feature_config):
    bonuses = feature_config["rerank_bonus"]
    reranked = []

    for candidate_position, distance in candidate_pairs:
        candidate_row = candidate_frame.iloc[candidate_position]
        similarity = 1 - float(distance)
        score = similarity

        same_destination = candidate_row["destination_norm"] == seed_row["destination_norm"]
        same_destination_group = candidate_row["destination_group"] == seed_row["destination_group"]
        same_activity = candidate_row["activity_type_mapped"] == seed_row["activity_type_mapped"]
        same_category = candidate_row["category_mapped"] == seed_row["category_mapped"]
        same_difficulty = candidate_row["difficulty_level_norm"] == seed_row["difficulty_level_norm"]

        if same_destination:
            score += bonuses["destination"]
        elif same_destination_group:
            score += bonuses["destination_group"]

        if same_activity:
            score += bonuses["activity"]
        if same_category:
            score += bonuses["category"]
        if same_difficulty:
            score += bonuses["difficulty"]

        reranked.append((candidate_position, float(distance), float(score)))

    reranked.sort(key=lambda row: row[2], reverse=True)
    return reranked

def recommend_from_query_vector(
    query_vector,
    seed_row,
    candidate_frame,
    knn_model,
    feature_config,
    retrieval_path,
    top_k=None,
    candidate_pool=None,
    exclude_item_id=None,
):
    top_k = top_k or feature_config["top_k"]
    candidate_pool = candidate_pool or feature_config["candidate_pool"]
    n_neighbors = min(len(candidate_frame), candidate_pool + (1 if exclude_item_id is not None else 0))

    distances, indices = knn_model.kneighbors(query_vector, n_neighbors=n_neighbors)

    candidate_pairs = []
    for candidate_position, distance in zip(indices[0], distances[0]):
        candidate_item_id = candidate_frame.iloc[candidate_position]["item_id"]
        if exclude_item_id is not None and candidate_item_id == exclude_item_id:
            continue
        candidate_pairs.append((candidate_position, float(distance)))
        if len(candidate_pairs) >= candidate_pool:
            break

    reranked = rerank_candidates(seed_row, candidate_pairs, candidate_frame, feature_config)[:top_k]

    rows = []
    for candidate_position, distance, score in reranked:
        candidate_row = candidate_frame.iloc[candidate_position]
        rows.append(
            {
                "query_item_id": seed_row["item_id"],
                "recommended_item_id": candidate_row["item_id"],
                "title": candidate_row["title"],
                "destination": candidate_row["destination"],
                "destination_norm": candidate_row["destination_norm"],
                "destination_group": candidate_row["destination_group"],
                "category": candidate_row["category"],
                "category_mapped": candidate_row["category_mapped"],
                "activity_type": candidate_row["activity_type"],
                "activity_type_mapped": candidate_row["activity_type_mapped"],
                "difficulty_level": candidate_row["difficulty_level"],
                "duration_text": candidate_row["duration_text"],
                "cosine_similarity": 1 - float(distance),
                "reranked_score": float(score),
                "candidate_row_position": int(candidate_position),
                "retrieval_path": retrieval_path,
            }
        )

    return pd.DataFrame(rows)

def recommend_from_matrix(
    query_items,
    query_matrix,
    query_row_position,
    candidate_frame,
    knn_model,
    feature_config,
    retrieval_path,
    top_k=None,
    candidate_pool=None,
    exclude_self=False,
):
    seed_row = query_items.iloc[query_row_position]
    query_vector = query_matrix[query_row_position : query_row_position + 1]
    exclude_item_id = seed_row["item_id"] if exclude_self else None
    return recommend_from_query_vector(
        query_vector=query_vector,
        seed_row=seed_row,
        candidate_frame=candidate_frame,
        knn_model=knn_model,
        feature_config=feature_config,
        retrieval_path=retrieval_path,
        top_k=top_k,
        candidate_pool=candidate_pool,
        exclude_item_id=exclude_item_id,
    )


# Evaluation Helpers

The dense path should preserve the spirit of the sparse notebook recommender before Django uses the exported vectors for `pgvector` retrieval. The comparison below therefore reports both path-level quality metrics and overlap between sparse and dense recommendation sets.


In [11]:
def safe_mean(series):
    cleaned = pd.Series(series).dropna()
    if cleaned.empty:
        return None
    return float(cleaned.mean())

def get_relevant_item_ids(seed_row_position, frame):
    seed_row = frame.iloc[seed_row_position]

    same_activity = frame["activity_type_mapped"] == seed_row["activity_type_mapped"]
    same_category = frame["category_mapped"] == seed_row["category_mapped"]
    same_destination = frame["destination_norm"] == seed_row["destination_norm"]
    same_destination_group = frame["destination_group"] == seed_row["destination_group"]
    same_difficulty = frame["difficulty_level_norm"] == seed_row["difficulty_level_norm"]
    same_duration = frame["duration_type_mapped"] == seed_row["duration_type_mapped"]

    relevance_mask = (
        same_activity
        | (same_category & same_destination)
        | (same_category & same_destination_group)
        | (same_category & same_difficulty & same_duration)
    )

    relevant_item_ids = set(frame.loc[relevance_mask, "item_id"].tolist())
    relevant_item_ids.discard(seed_row["item_id"])
    return relevant_item_ids

def compute_precision_at_k(recommended_item_ids, relevant_item_ids):
    if not recommended_item_ids:
        return 0.0
    hits = sum(item_id in relevant_item_ids for item_id in recommended_item_ids)
    return hits / len(recommended_item_ids)

def compute_recall_at_k(recommended_item_ids, relevant_item_ids):
    if not relevant_item_ids:
        return np.nan
    hits = sum(item_id in relevant_item_ids for item_id in recommended_item_ids)
    return hits / len(relevant_item_ids)

def compute_match_rate(recommendations_df, seed_row, column_name):
    if recommendations_df.empty:
        return 0.0
    return float((recommendations_df[column_name] == seed_row[column_name]).mean())

def compute_average_cosine_similarity(recommendations_df):
    if recommendations_df.empty:
        return 0.0
    return float(recommendations_df["cosine_similarity"].mean())

def compute_overlap_at_k(item_ids_a, item_ids_b, top_k):
    overlap_denominator = max(1, min(top_k, len(item_ids_a), len(item_ids_b)))
    return len(set(item_ids_a[:top_k]).intersection(item_ids_b[:top_k])) / overlap_denominator

def summarize_path_metrics(metrics_df):
    return {
        "n_evaluated_items": int(len(metrics_df)),
        "precision_at_k_mean": safe_mean(metrics_df["precision_at_k"]),
        "recall_at_k_mean": safe_mean(metrics_df["recall_at_k"]),
        "avg_cosine_similarity_at_k_mean": safe_mean(metrics_df["avg_cosine_similarity_at_k"]),
        "category_match_rate_at_k_mean": safe_mean(metrics_df["category_match_rate_at_k"]),
        "activity_type_match_rate_at_k_mean": safe_mean(metrics_df["activity_type_match_rate_at_k"]),
        "destination_match_rate_at_k_mean": safe_mean(metrics_df["destination_match_rate_at_k"]),
        "destination_group_match_rate_at_k_mean": safe_mean(metrics_df["destination_group_match_rate_at_k"]),
    }

def evaluate_single_path(
    frame,
    feature_matrix,
    knn_model,
    feature_config,
    path_label,
    top_k=5,
    sample_size=None,
    random_state=MODEL_RANDOM_STATE,
):
    row_positions = list(range(len(frame)))

    if sample_size is not None and sample_size < len(row_positions):
        rng = np.random.default_rng(random_state)
        row_positions = sorted(rng.choice(row_positions, size=sample_size, replace=False).tolist())

    rows = []
    recommendation_cache = {}

    for row_position in row_positions:
        seed_row = frame.iloc[row_position]
        recommendations_df = recommend_from_matrix(
            query_items=frame,
            query_matrix=feature_matrix,
            query_row_position=row_position,
            candidate_frame=frame,
            knn_model=knn_model,
            feature_config=feature_config,
            retrieval_path=path_label,
            top_k=top_k,
            exclude_self=True,
        )
        recommendation_cache[row_position] = recommendations_df

        recommended_item_ids = recommendations_df["recommended_item_id"].tolist()
        relevant_item_ids = get_relevant_item_ids(row_position, frame)

        rows.append(
            {
                "seed_row_position": int(row_position),
                "seed_item_id": seed_row["item_id"],
                "seed_title": seed_row["title"],
                "precision_at_k": compute_precision_at_k(recommended_item_ids, relevant_item_ids),
                "recall_at_k": compute_recall_at_k(recommended_item_ids, relevant_item_ids),
                "avg_cosine_similarity_at_k": compute_average_cosine_similarity(recommendations_df),
                "category_match_rate_at_k": compute_match_rate(recommendations_df, seed_row, "category_mapped"),
                "activity_type_match_rate_at_k": compute_match_rate(recommendations_df, seed_row, "activity_type_mapped"),
                "destination_match_rate_at_k": compute_match_rate(recommendations_df, seed_row, "destination_norm"),
                "destination_group_match_rate_at_k": compute_match_rate(recommendations_df, seed_row, "destination_group"),
            }
        )

    metrics_df = pd.DataFrame(rows)
    summary = summarize_path_metrics(metrics_df)
    return metrics_df, summary, recommendation_cache

def compare_sparse_and_dense_paths(
    frame,
    sparse_matrix,
    sparse_knn,
    dense_matrix,
    dense_knn,
    feature_config,
    top_k=5,
    sample_size=None,
    random_state=MODEL_RANDOM_STATE,
):
    sparse_metrics_df, sparse_summary, sparse_cache = evaluate_single_path(
        frame=frame,
        feature_matrix=sparse_matrix,
        knn_model=sparse_knn,
        feature_config=feature_config,
        path_label="sparse_offline",
        top_k=top_k,
        sample_size=sample_size,
        random_state=random_state,
    )

    dense_metrics_df, dense_summary, dense_cache = evaluate_single_path(
        frame=frame,
        feature_matrix=dense_matrix,
        knn_model=dense_knn,
        feature_config=feature_config,
        path_label="dense_pgvector_bridge",
        top_k=top_k,
        sample_size=sample_size,
        random_state=random_state,
    )

    comparison_df = sparse_metrics_df.merge(
        dense_metrics_df,
        on=["seed_row_position", "seed_item_id", "seed_title"],
        suffixes=("_sparse", "_dense"),
    )
    comparison_df["overlap_at_k"] = comparison_df["seed_row_position"].apply(
        lambda row_position: compute_overlap_at_k(
            sparse_cache[row_position]["recommended_item_id"].tolist(),
            dense_cache[row_position]["recommended_item_id"].tolist(),
            top_k=top_k,
        )
    )

    comparison_summary = {
        "top_k": int(top_k),
        "n_evaluated_items": int(len(comparison_df)),
        "sparse": sparse_summary,
        "dense": dense_summary,
        "dense_vs_sparse": {
            "overlap_at_k_mean": safe_mean(comparison_df["overlap_at_k"]),
            "precision_delta_mean": float(
                comparison_df["precision_at_k_dense"].mean() - comparison_df["precision_at_k_sparse"].mean()
            ),
            "category_match_delta_mean": float(
                comparison_df["category_match_rate_at_k_dense"].mean()
                - comparison_df["category_match_rate_at_k_sparse"].mean()
            ),
            "activity_type_match_delta_mean": float(
                comparison_df["activity_type_match_rate_at_k_dense"].mean()
                - comparison_df["activity_type_match_rate_at_k_sparse"].mean()
            ),
            "destination_match_delta_mean": float(
                comparison_df["destination_match_rate_at_k_dense"].mean()
                - comparison_df["destination_match_rate_at_k_sparse"].mean()
            ),
            "destination_group_match_delta_mean": float(
                comparison_df["destination_group_match_rate_at_k_dense"].mean()
                - comparison_df["destination_group_match_rate_at_k_sparse"].mean()
            ),
            "avg_cosine_similarity_delta_mean": float(
                comparison_df["avg_cosine_similarity_at_k_dense"].mean()
                - comparison_df["avg_cosine_similarity_at_k_sparse"].mean()
            ),
        },
    }

    return comparison_df, comparison_summary, sparse_cache, dense_cache

def summarize_query_set_against_training(
    query_items,
    query_matrix,
    candidate_items,
    knn_model,
    feature_config,
    path_label,
    top_k=5,
):
    rows = []
    recommendation_cache = {}

    for row_position in range(len(query_items)):
        seed_row = query_items.iloc[row_position]
        recommendations_df = recommend_from_matrix(
            query_items=query_items,
            query_matrix=query_matrix,
            query_row_position=row_position,
            candidate_frame=candidate_items,
            knn_model=knn_model,
            feature_config=feature_config,
            retrieval_path=path_label,
            top_k=top_k,
            exclude_self=False,
        )
        recommendation_cache[row_position] = recommendations_df

        rows.append(
            {
                "query_row_position": int(row_position),
                "query_item_id": seed_row["item_id"],
                "query_title": seed_row["title"],
                "category_match_rate_at_k": compute_match_rate(recommendations_df, seed_row, "category_mapped"),
                "activity_type_match_rate_at_k": compute_match_rate(recommendations_df, seed_row, "activity_type_mapped"),
                "destination_match_rate_at_k": compute_match_rate(recommendations_df, seed_row, "destination_norm"),
                "destination_group_match_rate_at_k": compute_match_rate(recommendations_df, seed_row, "destination_group"),
                "avg_cosine_similarity_at_k": compute_average_cosine_similarity(recommendations_df),
            }
        )

    metrics_df = pd.DataFrame(rows)
    summary = {
        "n_queries": int(len(metrics_df)),
        "category_match_rate_at_k_mean": safe_mean(metrics_df["category_match_rate_at_k"]),
        "activity_type_match_rate_at_k_mean": safe_mean(metrics_df["activity_type_match_rate_at_k"]),
        "destination_match_rate_at_k_mean": safe_mean(metrics_df["destination_match_rate_at_k"]),
        "destination_group_match_rate_at_k_mean": safe_mean(metrics_df["destination_group_match_rate_at_k"]),
        "avg_cosine_similarity_at_k_mean": safe_mean(metrics_df["avg_cosine_similarity_at_k"]),
    }
    return metrics_df, summary, recommendation_cache

def compare_query_sets_against_training(
    query_items,
    sparse_query_matrix,
    dense_query_matrix,
    candidate_items,
    sparse_knn,
    dense_knn,
    feature_config,
    top_k=5,
):
    sparse_metrics_df, sparse_summary, sparse_cache = summarize_query_set_against_training(
        query_items=query_items,
        query_matrix=sparse_query_matrix,
        candidate_items=candidate_items,
        knn_model=sparse_knn,
        feature_config=feature_config,
        path_label="sparse_offline",
        top_k=top_k,
    )
    dense_metrics_df, dense_summary, dense_cache = summarize_query_set_against_training(
        query_items=query_items,
        query_matrix=dense_query_matrix,
        candidate_items=candidate_items,
        knn_model=dense_knn,
        feature_config=feature_config,
        path_label="dense_pgvector_bridge",
        top_k=top_k,
    )

    comparison_df = sparse_metrics_df.merge(
        dense_metrics_df,
        on=["query_row_position", "query_item_id", "query_title"],
        suffixes=("_sparse", "_dense"),
    )
    comparison_df["overlap_at_k"] = comparison_df["query_row_position"].apply(
        lambda row_position: compute_overlap_at_k(
            sparse_cache[row_position]["recommended_item_id"].tolist(),
            dense_cache[row_position]["recommended_item_id"].tolist(),
            top_k=top_k,
        )
    )

    comparison_summary = {
        "top_k": int(top_k),
        "n_queries": int(len(comparison_df)),
        "sparse": sparse_summary,
        "dense": dense_summary,
        "dense_vs_sparse": {
            "overlap_at_k_mean": safe_mean(comparison_df["overlap_at_k"]),
            "category_match_delta_mean": float(
                comparison_df["category_match_rate_at_k_dense"].mean()
                - comparison_df["category_match_rate_at_k_sparse"].mean()
            ),
            "activity_type_match_delta_mean": float(
                comparison_df["activity_type_match_rate_at_k_dense"].mean()
                - comparison_df["activity_type_match_rate_at_k_sparse"].mean()
            ),
            "destination_match_delta_mean": float(
                comparison_df["destination_match_rate_at_k_dense"].mean()
                - comparison_df["destination_match_rate_at_k_sparse"].mean()
            ),
            "destination_group_match_delta_mean": float(
                comparison_df["destination_group_match_rate_at_k_dense"].mean()
                - comparison_df["destination_group_match_rate_at_k_sparse"].mean()
            ),
            "avg_cosine_similarity_delta_mean": float(
                comparison_df["avg_cosine_similarity_at_k_dense"].mean()
                - comparison_df["avg_cosine_similarity_at_k_sparse"].mean()
            ),
        },
    }

    return comparison_df, comparison_summary, sparse_cache, dense_cache


# Hyperparameter Optimization With Optuna

The tuning step still optimizes the notebook-side sparse debug recommender because that is the closest continuity point with the original workflow. The dense path is evaluated separately to confirm that the production bridge preserves recommendation behavior.


In [12]:
def objective(trial):
    trial_feature_config = make_feature_config(
        {
            "weights": {
                "tfidf": trial.suggest_float("W_TFIDF", 0.7, 1.5),
                "categorical": trial.suggest_float("W_CATEGORICAL", 0.4, 1.2),
                "multi_value": trial.suggest_float("W_MULTI_VALUE", 0.2, 1.0),
                "numeric": trial.suggest_float("W_NUMERIC", 0.1, 0.8),
            },
            "rerank_bonus": {
                "destination": trial.suggest_float("BONUS_DESTINATION", 0.10, 0.35),
                "destination_group": trial.suggest_float("BONUS_DESTINATION_GROUP", 0.05, 0.20),
                "activity": trial.suggest_float("BONUS_ACTIVITY", 0.05, 0.20),
                "category": trial.suggest_float("BONUS_CATEGORY", 0.03, 0.15),
                "difficulty": trial.suggest_float("BONUS_DIFFICULTY", 0.00, 0.10),
            },
            "candidate_pool": trial.suggest_int("CANDIDATE_POOL", 15, 30),
            "top_k": 5,
        }
    )

    feature_components = fit_feature_components(train_items, trial_feature_config)
    X_train_sparse_trial = transform_items_to_sparse(train_items, feature_components, trial_feature_config)
    sparse_knn_trial = fit_knn_index(X_train_sparse_trial)

    _, summary, _ = evaluate_single_path(
        frame=train_items,
        feature_matrix=X_train_sparse_trial,
        knn_model=sparse_knn_trial,
        feature_config=trial_feature_config,
        path_label="sparse_offline",
        top_k=trial_feature_config["top_k"],
        sample_size=min(300, len(train_items)),
        random_state=MODEL_RANDOM_STATE,
    )

    score = (
        0.35 * summary["precision_at_k_mean"]
        + 0.25 * summary["activity_type_match_rate_at_k_mean"]
        + 0.20 * summary["category_match_rate_at_k_mean"]
        + 0.20 * summary["destination_group_match_rate_at_k_mean"]
    )
    return score


In [13]:
# Reduce n_trials if runtime is too high on your machine.
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Best trial score:", study.best_value)
print("Best params:")
for key, value in study.best_params.items():
    print(f"{key}: {value}")


[I 2026-03-23 20:18:22,016] A new study created in memory with name: no-name-895b2cdb-fa73-418c-8f1a-b7b881c7918d
[I 2026-03-23 20:18:23,828] Trial 0 finished with value: 0.8750333333333333 and parameters: {'W_TFIDF': 0.9183100841923578, 'W_CATEGORICAL': 1.1564256756660078, 'W_MULTI_VALUE': 0.6254298225448028, 'W_NUMERIC': 0.6484647232529039, 'BONUS_DESTINATION': 0.20031875165025018, 'BONUS_DESTINATION_GROUP': 0.054572812809369105, 'BONUS_ACTIVITY': 0.05760056123527497, 'BONUS_CATEGORY': 0.03718920171654693, 'BONUS_DIFFICULTY': 0.02314458213090197, 'CANDIDATE_POOL': 26}. Best is trial 0 with value: 0.8750333333333333.
[I 2026-03-23 20:18:25,695] Trial 1 finished with value: 0.8417666666666668 and parameters: {'W_TFIDF': 1.244679884040336, 'W_CATEGORICAL': 0.5493208227281686, 'W_MULTI_VALUE': 0.6052987649718155, 'W_NUMERIC': 0.6149489362994729, 'BONUS_DESTINATION': 0.2818783606776554, 'BONUS_DESTINATION_GROUP': 0.06878379967458995, 'BONUS_ACTIVITY': 0.16699895236671836, 'BONUS_CATEGORY'

Best trial score: 0.9073333333333333
Best params:
W_TFIDF: 1.4592529880253635
W_CATEGORICAL: 0.7662404456689167
W_MULTI_VALUE: 0.2396666328698741
W_NUMERIC: 0.12294392064842627
BONUS_DESTINATION: 0.16474918123208132
BONUS_DESTINATION_GROUP: 0.15493899246850254
BONUS_ACTIVITY: 0.1925141926391777
BONUS_CATEGORY: 0.12298231542068731
BONUS_DIFFICULTY: 0.012660891537464357
CANDIDATE_POOL: 28


In [14]:
best_feature_config = make_feature_config()

if "study" in globals() and hasattr(study, "best_params"):
    best_feature_config = make_feature_config(
        {
            "weights": {
                "tfidf": study.best_params["W_TFIDF"],
                "categorical": study.best_params["W_CATEGORICAL"],
                "multi_value": study.best_params["W_MULTI_VALUE"],
                "numeric": study.best_params["W_NUMERIC"],
            },
            "rerank_bonus": {
                "destination": study.best_params["BONUS_DESTINATION"],
                "destination_group": study.best_params["BONUS_DESTINATION_GROUP"],
                "activity": study.best_params["BONUS_ACTIVITY"],
                "category": study.best_params["BONUS_CATEGORY"],
                "difficulty": study.best_params["BONUS_DIFFICULTY"],
            },
            "candidate_pool": study.best_params["CANDIDATE_POOL"],
            "top_k": 5,
        }
    )

print("Final feature config:")
print(json.dumps(make_json_safe(best_feature_config), indent=2))


Final feature config:
{
  "weights": {
    "tfidf": 1.4592529880253635,
    "categorical": 0.7662404456689167,
    "multi_value": 0.2396666328698741,
    "numeric": 0.12294392064842627
  },
  "rerank_bonus": {
    "destination": 0.16474918123208132,
    "destination_group": 0.15493899246850254,
    "activity": 0.1925141926391777,
    "category": 0.12298231542068731,
    "difficulty": 0.012660891537464357
  },
  "candidate_pool": 28,
  "top_k": 5,
  "tfidf": {
    "max_features": 10000,
    "ngram_range": [
      1,
      2
    ],
    "min_df": 3,
    "max_df": 0.85,
    "sublinear_tf": true,
    "stop_words": "english"
  },
  "dense_projection": {
    "target_dim": 128,
    "normalization": "l2",
    "random_state": 42
  }
}


# Final Training: Sparse Debug Path And Dense Production Path

The fitted sparse matrix and sparse KNN below remain notebook-only debug tools. The fitted dense projector and normalized dense vectors are the production-facing representation that later Django code can generate again for live listings before handing retrieval to `pgvector`.


In [15]:
final_feature_components = fit_feature_components(train_items, best_feature_config)
X_train_sparse = transform_items_to_sparse(train_items, final_feature_components, best_feature_config)
final_dense_projector, X_train_dense = fit_dense_projector(X_train_sparse, best_feature_config)

sparse_knn = fit_knn_index(X_train_sparse)
dense_knn = fit_knn_index(X_train_dense)

print("Sparse feature matrix shape:", X_train_sparse.shape)
print("Dense training vector shape:", X_train_dense.shape)
print("Dense explained variance ratio sum:", float(final_dense_projector.explained_variance_ratio_.sum()))


Sparse feature matrix shape: (1440, 10775)
Dense training vector shape: (1440, 128)
Dense explained variance ratio sum: 0.7344008470131698


# Dense Path Evaluation

The dense projection is only useful for backend integration if it keeps the same recommendation intent as the sparse notebook recommender. The evaluation below compares the two paths on precision, category and activity alignment, destination behavior, cosine similarity, and recommendation overlap.


In [16]:
comparison_metrics_df, comparison_summary, sparse_train_cache, dense_train_cache = compare_sparse_and_dense_paths(
    frame=train_items,
    sparse_matrix=X_train_sparse,
    sparse_knn=sparse_knn,
    dense_matrix=X_train_dense,
    dense_knn=dense_knn,
    feature_config=best_feature_config,
    top_k=best_feature_config["top_k"],
)

comparison_summary_display = pd.DataFrame(
    [
        {"path": "sparse_offline", **comparison_summary["sparse"]},
        {"path": "dense_pgvector_bridge", **comparison_summary["dense"]},
        {"path": "dense_vs_sparse", **comparison_summary["dense_vs_sparse"]},
    ]
)

display(comparison_summary_display)
display(comparison_metrics_df.head(10))


,path,n_evaluated_items,precision_at_k_mean,recall_at_k_mean,avg_cosine_similarity_at_k_mean,category_match_rate_at_k_mean,activity_type_match_rate_at_k_mean,destination_match_rate_at_k_mean,destination_group_match_rate_at_k_mean,overlap_at_k_mean,precision_delta_mean,category_match_delta_mean,activity_type_match_delta_mean,destination_match_delta_mean,destination_group_match_delta_mean,avg_cosine_similarity_delta_mean
0,sparse_offline,1440.0,0.997778,0.041629,0.645832,0.896111,0.967778,0.629444,0.672500,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,dense_pgvector_bridge,1440.0,0.998333,0.041649,0.826104,0.906806,0.971944,0.606667,0.648333,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,dense_vs_sparse,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.708333,0.000556,0.010694,0.004167,-0.022778,-0.024167,0.180272


,seed_row_position,seed_item_id,seed_title,precision_at_k_sparse,recall_at_k_sparse,avg_cosine_similarity_at_k_sparse,category_match_rate_at_k_sparse,activity_type_match_rate_at_k_sparse,destination_match_rate_at_k_sparse,destination_group_match_rate_at_k_sparse,precision_at_k_dense,recall_at_k_dense,avg_cosine_similarity_at_k_dense,category_match_rate_at_k_dense,activity_type_match_rate_at_k_dense,destination_match_rate_at_k_dense,destination_group_match_rate_at_k_dense,overlap_at_k
0,0,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,Kathmandu UNESCO Heritage Circuit with Private Guide,1.0,0.020921,0.637198,1.0,1.0,0.6,1.0,1.0,0.020921,0.911534,1.0,1.0,0.6,1.0,0.6
1,1,scraped_pashupatinath_boudhanath_and_patan_herit_c1faa00f8590b3d8,"Pashupatinath, Boudhanath and Patan Heritage Day",1.0,0.020921,0.639935,1.0,1.0,0.6,1.0,1.0,0.020921,0.913480,1.0,1.0,0.6,1.0,0.8
2,2,scraped_kathmandu_valley_unesco_sites_day_tour_bae70cfbba82963e,Kathmandu Valley UNESCO Sites Day Tour,1.0,0.020921,0.675338,1.0,1.0,1.0,1.0,1.0,0.020921,0.930261,1.0,1.0,1.0,1.0,0.6
3,3,scraped_bhaktapur_durbar_square_and_old_town_wal_28a5a611f448b797,Bhaktapur Durbar Square and Old Town Walk,1.0,0.021552,0.679257,1.0,1.0,0.6,1.0,1.0,0.021552,0.911155,1.0,1.0,0.6,1.0,0.8
4,4,scraped_patan_museum_golden_temple_and_courtyard_4256bea42560d249,"Patan Museum, Golden Temple and Courtyard Walk",1.0,0.021552,0.677176,1.0,1.0,1.0,1.0,1.0,0.021552,0.943217,1.0,1.0,0.8,1.0,0.8
5,5,scraped_thamel_asan_and_indra_chowk_bazaar_walk_f218b7fa14307a36,"Thamel, Asan and Indra Chowk Bazaar Walk",1.0,0.021552,0.639205,1.0,1.0,0.8,1.0,1.0,0.021552,0.938244,1.0,1.0,0.8,1.0,0.8
6,6,scraped_boudhanath_and_pashupatinath_spiritual_c_f24413cd34e583e8,Boudhanath and Pashupatinath Spiritual Circuit,1.0,0.026455,0.699431,1.0,1.0,1.0,1.0,1.0,0.026455,0.985788,1.0,1.0,1.0,1.0,1.0
7,7,scraped_bhaktapur_and_changu_narayan_heritage_da_e9f37a3a88c42b08,Bhaktapur and Changu Narayan Heritage Day Trip,1.0,0.023364,0.505921,0.8,1.0,0.2,1.0,1.0,0.023364,0.779006,0.4,1.0,0.6,1.0,0.6
8,8,scraped_patan_and_kirtipur_cultural_day_out_6439cfb58cb1c30d,Patan and Kirtipur Cultural Day Out,1.0,0.020921,0.615860,1.0,1.0,0.2,1.0,1.0,0.020921,0.851517,1.0,1.0,0.2,1.0,0.6
9,9,scraped_bhaktapur_and_patan_highlights_from_kath_34eb585502c955e3,Bhaktapur and Patan Highlights from Kathmandu,1.0,0.020921,0.652144,1.0,1.0,0.8,1.0,1.0,0.020921,0.927201,1.0,1.0,0.8,1.0,0.6


# Qualitative Recommendation Inspection


In [17]:
sample_seed_position = 0
sample_seed = train_items.iloc[sample_seed_position]

print("SEED:")
display(
    train_items.iloc[
        [sample_seed_position]
    ][
        [
            "item_id",
            "title",
            "destination",
            "category_mapped",
            "activity_type_mapped",
            "difficulty_level_norm",
            "duration_type_mapped",
        ]
    ]
)

print("\nSPARSE OFFLINE DEBUG PATH")
display(
    recommend_from_matrix(
        query_items=train_items,
        query_matrix=X_train_sparse,
        query_row_position=sample_seed_position,
        candidate_frame=train_items,
        knn_model=sparse_knn,
        feature_config=best_feature_config,
        retrieval_path="sparse_offline",
        top_k=best_feature_config["top_k"],
        exclude_self=True,
    )
)

print("\nDENSE PRODUCTION BRIDGE PATH")
display(
    recommend_from_matrix(
        query_items=train_items,
        query_matrix=X_train_dense,
        query_row_position=sample_seed_position,
        candidate_frame=train_items,
        knn_model=dense_knn,
        feature_config=best_feature_config,
        retrieval_path="dense_pgvector_bridge",
        top_k=best_feature_config["top_k"],
        exclude_self=True,
    )
)


SEED:


,item_id,title,destination,category_mapped,activity_type_mapped,difficulty_level_norm,duration_type_mapped
0,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,Kathmandu UNESCO Heritage Circuit with Private Guide,Kathmandu,heritage,city_tour,easy,full_day



SPARSE OFFLINE DEBUG PATH


,query_item_id,recommended_item_id,title,destination,destination_norm,destination_group,category,category_mapped,activity_type,activity_type_mapped,difficulty_level,duration_text,cosine_similarity,reranked_score,candidate_row_position,retrieval_path
0,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,scraped_kathmandu_seven_unesco_heritage_sites_to_ccc5c90b91651ec4,Kathmandu Seven UNESCO Heritage Sites Tour,Kathmandu,kathmandu,kathmandu_valley,Culture | Landmarks,heritage,CITY_TOUR,city_tour,EASY,9 hours,0.705514,1.198420,18,sparse_offline
1,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,scraped_pashupatinath_boudhanath_and_patan_herit_c1faa00f8590b3d8,"Pashupatinath, Boudhanath and Patan Heritage Day",Kathmandu,kathmandu,kathmandu_valley,Old towns | Guided touring,heritage,CITY_TOUR,city_tour,EASY,5.5 hours,0.679049,1.171956,1,sparse_offline
2,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,scraped_chobar_gorge_kirtipur_and_taudaha_half_d_c602a12f2c95862b,"Chobar Gorge, Kirtipur and Taudaha Half Day",Kathmandu,kathmandu,kathmandu_valley,Heritage | Local neighborhoods,heritage,CITY_TOUR,city_tour,EASY,8 hours,0.610574,1.103480,1091,sparse_offline
3,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,scraped_kathmandu_valley_unesco_sites_day_tour_bae70cfbba82963e,Kathmandu Valley UNESCO Sites Day Tour,Kathmandu Valley,kathmandu valley,kathmandu_valley,Old towns | Guided touring,heritage,CITY_TOUR,city_tour,EASY,8 hours,0.601027,1.084123,2,sparse_offline
4,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,scraped_bungamati_and_khokana_newari_craft_trail_d47dfd3be7e14d5c,Bungamati and Khokana Newari Craft Trail,Kathmandu Valley,kathmandu valley,kathmandu_valley,Culture | Landmarks,heritage,CITY_TOUR,city_tour,EASY,5.5 hours,0.589825,1.072921,1203,sparse_offline



DENSE PRODUCTION BRIDGE PATH


,query_item_id,recommended_item_id,title,destination,destination_norm,destination_group,category,category_mapped,activity_type,activity_type_mapped,difficulty_level,duration_text,cosine_similarity,reranked_score,candidate_row_position,retrieval_path
0,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,scraped_kathmandu_seven_unesco_heritage_sites_to_ccc5c90b91651ec4,Kathmandu Seven UNESCO Heritage Sites Tour,Kathmandu,kathmandu,kathmandu_valley,Culture | Landmarks,heritage,CITY_TOUR,city_tour,EASY,9 hours,0.979773,1.472679,18,dense_pgvector_bridge
1,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,scraped_pashupatinath_boudhanath_and_patan_herit_c1faa00f8590b3d8,"Pashupatinath, Boudhanath and Patan Heritage Day",Kathmandu,kathmandu,kathmandu_valley,Old towns | Guided touring,heritage,CITY_TOUR,city_tour,EASY,5.5 hours,0.977488,1.470395,1,dense_pgvector_bridge
2,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,scraped_chobar_gorge_kirtipur_and_taudaha_half_d_c602a12f2c95862b,"Chobar Gorge, Kirtipur and Taudaha Half Day",Kathmandu,kathmandu,kathmandu_valley,Heritage | Local neighborhoods,heritage,CITY_TOUR,city_tour,EASY,8 hours,0.871371,1.364277,1091,dense_pgvector_bridge
3,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,scraped_bhaktapur_and_patan_highlights_from_kath_34eb585502c955e3,Bhaktapur and Patan Highlights from Kathmandu,Kathmandu Valley,kathmandu valley,kathmandu_valley,Old towns | Guided touring,heritage,CITY_TOUR,city_tour,EASY,8.5 hours,0.867502,1.350598,9,dense_pgvector_bridge
4,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,scraped_patan_and_bhaktapur_full_day_sightseeing_0a8166ea63fa2a89,Patan and Bhaktapur Full-Day Sightseeing,Kathmandu Valley,kathmandu valley,kathmandu_valley,Culture | Landmarks,heritage,CITY_TOUR,city_tour,EASY,7 hours,0.861538,1.344634,17,dense_pgvector_bridge


In [18]:
sample_row_positions = [row for row in [0, 100, 300, 700, 1000] if row < len(train_items)]

for row_position in sample_row_positions:
    seed_row = train_items.iloc[row_position]
    print("=" * 100)
    print("SEED:", seed_row["title"], "|", seed_row["item_id"])
    display(
        train_items.iloc[
            [row_position]
        ][
            [
                "destination",
                "category_mapped",
                "activity_type_mapped",
                "difficulty_level_norm",
                "duration_type_mapped",
            ]
        ]
    )
    print("Sparse offline debug recommendations")
    display(
        recommend_from_matrix(
            query_items=train_items,
            query_matrix=X_train_sparse,
            query_row_position=row_position,
            candidate_frame=train_items,
            knn_model=sparse_knn,
            feature_config=best_feature_config,
            retrieval_path="sparse_offline",
            top_k=best_feature_config["top_k"],
            exclude_self=True,
        )
    )
    print("Dense production bridge recommendations")
    display(
        recommend_from_matrix(
            query_items=train_items,
            query_matrix=X_train_dense,
            query_row_position=row_position,
            candidate_frame=train_items,
            knn_model=dense_knn,
            feature_config=best_feature_config,
            retrieval_path="dense_pgvector_bridge",
            top_k=best_feature_config["top_k"],
            exclude_self=True,
        )
    )


SEED: Kathmandu UNESCO Heritage Circuit with Private Guide | scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c


,destination,category_mapped,activity_type_mapped,difficulty_level_norm,duration_type_mapped
0,Kathmandu,heritage,city_tour,easy,full_day


Sparse offline debug recommendations


,query_item_id,recommended_item_id,title,destination,destination_norm,destination_group,category,category_mapped,activity_type,activity_type_mapped,difficulty_level,duration_text,cosine_similarity,reranked_score,candidate_row_position,retrieval_path
0,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,scraped_kathmandu_seven_unesco_heritage_sites_to_ccc5c90b91651ec4,Kathmandu Seven UNESCO Heritage Sites Tour,Kathmandu,kathmandu,kathmandu_valley,Culture | Landmarks,heritage,CITY_TOUR,city_tour,EASY,9 hours,0.705514,1.198420,18,sparse_offline
1,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,scraped_pashupatinath_boudhanath_and_patan_herit_c1faa00f8590b3d8,"Pashupatinath, Boudhanath and Patan Heritage Day",Kathmandu,kathmandu,kathmandu_valley,Old towns | Guided touring,heritage,CITY_TOUR,city_tour,EASY,5.5 hours,0.679049,1.171956,1,sparse_offline
2,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,scraped_chobar_gorge_kirtipur_and_taudaha_half_d_c602a12f2c95862b,"Chobar Gorge, Kirtipur and Taudaha Half Day",Kathmandu,kathmandu,kathmandu_valley,Heritage | Local neighborhoods,heritage,CITY_TOUR,city_tour,EASY,8 hours,0.610574,1.103480,1091,sparse_offline
3,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,scraped_kathmandu_valley_unesco_sites_day_tour_bae70cfbba82963e,Kathmandu Valley UNESCO Sites Day Tour,Kathmandu Valley,kathmandu valley,kathmandu_valley,Old towns | Guided touring,heritage,CITY_TOUR,city_tour,EASY,8 hours,0.601027,1.084123,2,sparse_offline
4,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,scraped_bungamati_and_khokana_newari_craft_trail_d47dfd3be7e14d5c,Bungamati and Khokana Newari Craft Trail,Kathmandu Valley,kathmandu valley,kathmandu_valley,Culture | Landmarks,heritage,CITY_TOUR,city_tour,EASY,5.5 hours,0.589825,1.072921,1203,sparse_offline


Dense production bridge recommendations


,query_item_id,recommended_item_id,title,destination,destination_norm,destination_group,category,category_mapped,activity_type,activity_type_mapped,difficulty_level,duration_text,cosine_similarity,reranked_score,candidate_row_position,retrieval_path
0,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,scraped_kathmandu_seven_unesco_heritage_sites_to_ccc5c90b91651ec4,Kathmandu Seven UNESCO Heritage Sites Tour,Kathmandu,kathmandu,kathmandu_valley,Culture | Landmarks,heritage,CITY_TOUR,city_tour,EASY,9 hours,0.979773,1.472679,18,dense_pgvector_bridge
1,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,scraped_pashupatinath_boudhanath_and_patan_herit_c1faa00f8590b3d8,"Pashupatinath, Boudhanath and Patan Heritage Day",Kathmandu,kathmandu,kathmandu_valley,Old towns | Guided touring,heritage,CITY_TOUR,city_tour,EASY,5.5 hours,0.977488,1.470395,1,dense_pgvector_bridge
2,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,scraped_chobar_gorge_kirtipur_and_taudaha_half_d_c602a12f2c95862b,"Chobar Gorge, Kirtipur and Taudaha Half Day",Kathmandu,kathmandu,kathmandu_valley,Heritage | Local neighborhoods,heritage,CITY_TOUR,city_tour,EASY,8 hours,0.871371,1.364277,1091,dense_pgvector_bridge
3,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,scraped_bhaktapur_and_patan_highlights_from_kath_34eb585502c955e3,Bhaktapur and Patan Highlights from Kathmandu,Kathmandu Valley,kathmandu valley,kathmandu_valley,Old towns | Guided touring,heritage,CITY_TOUR,city_tour,EASY,8.5 hours,0.867502,1.350598,9,dense_pgvector_bridge
4,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,scraped_patan_and_bhaktapur_full_day_sightseeing_0a8166ea63fa2a89,Patan and Bhaktapur Full-Day Sightseeing,Kathmandu Valley,kathmandu valley,kathmandu_valley,Culture | Landmarks,heritage,CITY_TOUR,city_tour,EASY,7 hours,0.861538,1.344634,17,dense_pgvector_bridge


... (20 outputs left out)

Sparse offline debug recommendations


,query_item_id,recommended_item_id,title,destination,destination_norm,destination_group,category,category_mapped,activity_type,activity_type_mapped,difficulty_level,duration_text,cosine_similarity,reranked_score,candidate_row_position,retrieval_path
0,scraped_mai_pokhari_wetland_and_monastery_day_e81c378a6602e07e,scraped_mai_pokhari_nature_and_monastery_day_1fc587ecd2842d7f,Mai Pokhari Nature and Monastery Day,Ilam,ilam,ilam,Nature | Spiritual landscapes,nature,CITY_TOUR,city_tour,EASY,5 hours,0.777107,1.270014,516,sparse_offline
1,scraped_mai_pokhari_wetland_and_monastery_day_e81c378a6602e07e,scraped_ilam_tea_garden_and_kanyam_mist_day_3d982c68f52167e4,Ilam Tea Garden and Kanyam Mist Day,Ilam,ilam,ilam,Regional culture | Town walks,cultural,CITY_TOUR,city_tour,EASY,6 hours,0.548869,0.918793,1100,sparse_offline
2,scraped_mai_pokhari_wetland_and_monastery_day_e81c378a6602e07e,scraped_ilam_tea_garden_picnic_and_kanyam_ridge__bb005441fcc5577e,Ilam Tea Garden Picnic and Kanyam Ridge Ride,Ilam,ilam,ilam,Tea hills | Day outings,cultural,CITY_TOUR,city_tour,EASY,5 hours,0.536394,0.906319,272,sparse_offline
3,scraped_mai_pokhari_wetland_and_monastery_day_e81c378a6602e07e,scraped_pokhara_seti_gorge_cave_and_old_bazaar_h_7cd4ea69a012f672,"Pokhara Seti Gorge, Cave and Old Bazaar Half-Day",Pokhara,pokhara,pokhara_annapurna,Nature | City sights,nature,CITY_TOUR,city_tour,EASY,4 hours,0.524417,0.852574,258,sparse_offline
4,scraped_mai_pokhari_wetland_and_monastery_day_e81c378a6602e07e,scraped_ilam_tea_garden_sunrise_walk_with_local__e36111a65eb57655,Ilam Tea-Garden Sunrise Walk with Local Breakfast,Ilam,ilam,ilam,Nature | Guided walks,nature,HIKING,hiking,Easy,4.5 hours,0.526463,0.826856,887,sparse_offline


Dense production bridge recommendations


,query_item_id,recommended_item_id,title,destination,destination_norm,destination_group,category,category_mapped,activity_type,activity_type_mapped,difficulty_level,duration_text,cosine_similarity,reranked_score,candidate_row_position,retrieval_path
0,scraped_mai_pokhari_wetland_and_monastery_day_e81c378a6602e07e,scraped_mai_pokhari_nature_and_monastery_day_1fc587ecd2842d7f,Mai Pokhari Nature and Monastery Day,Ilam,ilam,ilam,Nature | Spiritual landscapes,nature,CITY_TOUR,city_tour,EASY,5 hours,0.905021,1.397928,516,dense_pgvector_bridge
1,scraped_mai_pokhari_wetland_and_monastery_day_e81c378a6602e07e,scraped_ilam_tea_garden_picnic_and_kanyam_ridge__bb005441fcc5577e,Ilam Tea Garden Picnic and Kanyam Ridge Ride,Ilam,ilam,ilam,Tea hills | Day outings,cultural,CITY_TOUR,city_tour,EASY,5 hours,0.704349,1.074273,272,dense_pgvector_bridge
2,scraped_mai_pokhari_wetland_and_monastery_day_e81c378a6602e07e,scraped_pokhara_seti_gorge_cave_and_old_bazaar_h_7cd4ea69a012f672,"Pokhara Seti Gorge, Cave and Old Bazaar Half-Day",Pokhara,pokhara,pokhara_annapurna,Nature | City sights,nature,CITY_TOUR,city_tour,EASY,4 hours,0.728628,1.056786,258,dense_pgvector_bridge
3,scraped_mai_pokhari_wetland_and_monastery_day_e81c378a6602e07e,scraped_ilam_tea_garden_sunrise_walk_with_local__e36111a65eb57655,Ilam Tea-Garden Sunrise Walk with Local Breakfast,Ilam,ilam,ilam,Nature | Guided walks,nature,HIKING,hiking,Easy,4.5 hours,0.717035,1.017427,887,dense_pgvector_bridge
4,scraped_mai_pokhari_wetland_and_monastery_day_e81c378a6602e07e,scraped_ilam_tea_garden_and_kanyam_mist_day_3d982c68f52167e4,Ilam Tea Garden and Kanyam Mist Day,Ilam,ilam,ilam,Regional culture | Town walks,cultural,CITY_TOUR,city_tour,EASY,6 hours,0.641390,1.011314,1100,dense_pgvector_bridge


# Unseen Item Testing With Shared Canonical Transform

Unseen evaluation now uses the exact same `prepare_items()` and transform functions as the scraped training data. This keeps the notebook-side logic aligned with the later Django preprocessing contract for live DB listings.


In [19]:
unseen_data = [
    {
        "title": "Kathmandu UNESCO Heritage Walk with Local Storytelling Guide",
        "description": "A guided cultural walking experience through old Kathmandu heritage lanes, palace courtyards, temple zones, and living urban history with strong local storytelling.",
        "destination": "Kathmandu",
        "category": "Culture | Landmarks",
        "activity_type": "CITY_TOUR",
        "difficulty_level": "EASY",
        "duration_type": "FULL_DAY",
        "duration_days": np.nan,
        "duration_nights": np.nan,
        "average_duration_hours": 6.5,
        "duration_text": "6.5 hours",
        "tags": "Kathmandu | UNESCO | heritage walk | temples | old city",
        "ideal_for": "First-time visitors | Couples | Solo travelers",
        "best_seasons": "YEAR_ROUND",
        "languages_offered": "English | Hindi",
        "max_group_size": 10,
        "instant_confirmation": True,
        "is_bookable": True,
        "is_active": True,
    },
    {
        "title": "Pokhara Lakeside Sunrise Paragliding Experience",
        "description": "A sunrise paragliding activity above Pokhara with lake views, mountain backdrop, and tandem flight support for first-time flyers.",
        "destination": "Pokhara",
        "category": "Adventure | Air activities",
        "activity_type": "PARAGLIDING",
        "difficulty_level": "EASY",
        "duration_type": "HALF_DAY",
        "duration_days": np.nan,
        "duration_nights": np.nan,
        "average_duration_hours": 3.0,
        "duration_text": "3 hours",
        "tags": "Pokhara | paragliding | sunrise flight | tandem",
        "ideal_for": "Adventure seekers | Couples | Solo travelers",
        "best_seasons": "Autumn | Spring",
        "languages_offered": "English",
        "max_group_size": 8,
        "instant_confirmation": True,
        "is_bookable": True,
        "is_active": True,
    },
    {
        "title": "Chitwan Jeep and Wildlife Observation Safari",
        "description": "A jungle wildlife outing in Chitwan combining jeep exploration, habitat observation, and chances to spot birds, deer, and other wildlife.",
        "destination": "Chitwan",
        "category": "Nature | Wildlife",
        "activity_type": "WILDLIFE_SAFARI",
        "difficulty_level": "EASY",
        "duration_type": "FULL_DAY",
        "duration_days": np.nan,
        "duration_nights": np.nan,
        "average_duration_hours": 7.0,
        "duration_text": "7 hours",
        "tags": "Chitwan | wildlife safari | jungle | jeep safari | birds",
        "ideal_for": "Families | Couples | Nature-focused travelers",
        "best_seasons": "Autumn | Winter | Spring",
        "languages_offered": "English",
        "max_group_size": 12,
        "instant_confirmation": True,
        "is_bookable": True,
        "is_active": True,
    },
    {
        "title": "Bhaktapur Pottery and Old Town Cultural Workshop",
        "description": "A cultural workshop in Bhaktapur combining pottery practice, old-town exploration, temple courtyards, and local artisan context.",
        "destination": "Bhaktapur",
        "category": "Old towns | Guided touring",
        "activity_type": "CULTURAL_WORKSHOP",
        "difficulty_level": "EASY",
        "duration_type": "HALF_DAY",
        "duration_days": np.nan,
        "duration_nights": np.nan,
        "average_duration_hours": 4.0,
        "duration_text": "4 hours",
        "tags": "Bhaktapur | pottery | heritage | old town | artisan",
        "ideal_for": "Culture-focused travelers | Families | Couples",
        "best_seasons": "YEAR_ROUND",
        "languages_offered": "English",
        "max_group_size": 10,
        "instant_confirmation": True,
        "is_bookable": True,
        "is_active": True,
    },
    {
        "title": "Nagarkot Morning Yoga with Himalayan Viewpoint Session",
        "description": "A short yoga and breathing session in Nagarkot designed around sunrise, mountain views, and a calm wellness-focused morning atmosphere.",
        "destination": "Nagarkot",
        "category": "Wellness | Viewpoint retreat",
        "activity_type": "YOGA",
        "difficulty_level": "EASY",
        "duration_type": "HALF_DAY",
        "duration_days": np.nan,
        "duration_nights": np.nan,
        "average_duration_hours": 2.5,
        "duration_text": "2.5 hours",
        "tags": "Nagarkot | yoga | sunrise | wellness | mountain view",
        "ideal_for": "Couples | Wellness travelers | Solo travelers",
        "best_seasons": "Spring | Autumn",
        "languages_offered": "English",
        "max_group_size": 12,
        "instant_confirmation": True,
        "is_bookable": True,
        "is_active": True,
    },
    {
        "title": "Bhote Koshi Extreme Bungee Adventure Day",
        "description": "An adrenaline-focused bungee day trip at Bhote Koshi with jump coordination, transport support, and a fast-paced adventure setup.",
        "destination": "Bhote Koshi",
        "category": "Adventure | Extreme sports",
        "activity_type": "BUNGEE",
        "difficulty_level": "MODERATE",
        "duration_type": "FULL_DAY",
        "duration_days": np.nan,
        "duration_nights": np.nan,
        "average_duration_hours": 8.0,
        "duration_text": "8 hours",
        "tags": "Bhote Koshi | bungee | adrenaline | extreme adventure",
        "ideal_for": "Adventure seekers | Groups | Solo travelers",
        "best_seasons": "Autumn | Spring",
        "languages_offered": "English",
        "max_group_size": 15,
        "instant_confirmation": True,
        "is_bookable": True,
        "is_active": True,
    },
    {
        "title": "Pokhara Nepali Cooking Class with Market Visit",
        "description": "A guided culinary experience in Pokhara including a local market visit, ingredient introduction, and hands-on cooking session.",
        "destination": "Pokhara",
        "category": "Food | Cultural workshop",
        "activity_type": "COOKING_CLASS",
        "difficulty_level": "EASY",
        "duration_type": "HALF_DAY",
        "duration_days": np.nan,
        "duration_nights": np.nan,
        "average_duration_hours": 4.5,
        "duration_text": "4.5 hours",
        "tags": "Pokhara | cooking class | food | local market | cuisine",
        "ideal_for": "Families | Couples | Culture-focused travelers",
        "best_seasons": "YEAR_ROUND",
        "languages_offered": "English",
        "max_group_size": 10,
        "instant_confirmation": True,
        "is_bookable": True,
        "is_active": True,
    },
    {
        "title": "Everest Region Scenic Helicopter Mountain Flight",
        "description": "A scenic mountain flight experience focused on Himalayan aerial views and a short premium air-tour style itinerary in the Everest region.",
        "destination": "Everest Region",
        "category": "Adventure | Scenic flight",
        "activity_type": "MOUNTAIN_FLIGHT",
        "difficulty_level": "EASY",
        "duration_type": "HALF_DAY",
        "duration_days": np.nan,
        "duration_nights": np.nan,
        "average_duration_hours": 3.5,
        "duration_text": "3.5 hours",
        "tags": "Everest Region | helicopter | mountain flight | aerial views",
        "ideal_for": "Couples | Luxury travelers | First-time visitors",
        "best_seasons": "Spring | Autumn",
        "languages_offered": "English",
        "max_group_size": 5,
        "instant_confirmation": True,
        "is_bookable": True,
        "is_active": True,
    },
]

unseen_raw_df = pd.DataFrame(unseen_data)
display(unseen_raw_df.head())


,title,description,destination,category,activity_type,difficulty_level,duration_type,duration_days,duration_nights,average_duration_hours,duration_text,tags,ideal_for,best_seasons,languages_offered,max_group_size,instant_confirmation,is_bookable,is_active
0,Kathmandu UNESCO Heritage Walk with Local Storytelling Guide,"A guided cultural walking experience through old Kathmandu heritage lanes, palace courtyards, temple zones, and living urban history with strong local storytelling.",Kathmandu,Culture | Landmarks,CITY_TOUR,EASY,FULL_DAY,NaN,NaN,6.5,6.5 hours,Kathmandu | UNESCO | heritage walk | temples | old city,First-time visitors | Couples | Solo travelers,YEAR_ROUND,English | Hindi,10,True,True,True
1,Pokhara Lakeside Sunrise Paragliding Experience,"A sunrise paragliding activity above Pokhara with lake views, mountain backdrop, and tandem flight support for first-time flyers.",Pokhara,Adventure | Air activities,PARAGLIDING,EASY,HALF_DAY,NaN,NaN,3.0,3 hours,Pokhara | paragliding | sunrise flight | tandem,Adventure seekers | Couples | Solo travelers,Autumn | Spring,English,8,True,True,True
2,Chitwan Jeep and Wildlife Observation Safari,"A jungle wildlife outing in Chitwan combining jeep exploration, habitat observation, and chances to spot birds, deer, and other wildlife.",Chitwan,Nature | Wildlife,WILDLIFE_SAFARI,EASY,FULL_DAY,NaN,NaN,7.0,7 hours,Chitwan | wildlife safari | jungle | jeep safari | birds,Families | Couples | Nature-focused travelers,Autumn | Winter | Spring,English,12,True,True,True
3,Bhaktapur Pottery and Old Town Cultural Workshop,"A cultural workshop in Bhaktapur combining pottery practice, old-town exploration, temple courtyards, and local artisan context.",Bhaktapur,Old towns | Guided touring,CULTURAL_WORKSHOP,EASY,HALF_DAY,NaN,NaN,4.0,4 hours,Bhaktapur | pottery | heritage | old town | artisan,Culture-focused travelers | Families | Couples,YEAR_ROUND,English,10,True,True,True
4,Nagarkot Morning Yoga with Himalayan Viewpoint Session,"A short yoga and breathing session in Nagarkot designed around sunrise, mountain views, and a calm wellness-focused morning atmosphere.",Nagarkot,Wellness | Viewpoint retreat,YOGA,EASY,HALF_DAY,NaN,NaN,2.5,2.5 hours,Nagarkot | yoga | sunrise | wellness | mountain view,Couples | Wellness travelers | Solo travelers,Spring | Autumn,English,12,True,True,True


In [20]:
unseen_items = prepare_items(unseen_raw_df, source_name="unseen")
X_unseen_sparse = transform_items_to_sparse(unseen_items, final_feature_components, best_feature_config)
X_unseen_dense = transform_items_to_dense(
    unseen_items,
    final_feature_components,
    final_dense_projector,
    best_feature_config,
)

prepared_single_item, single_dense_vector = transform_single_item_to_dense(
    unseen_data[0],
    final_feature_components,
    final_dense_projector,
    best_feature_config,
    source_name="unseen_single",
)

print("Unseen canonical shape:", unseen_items.shape)
print("Unseen sparse feature matrix shape:", X_unseen_sparse.shape)
print("Unseen dense vector shape:", X_unseen_dense.shape)
print("Single-item dense vector length:", len(single_dense_vector))

display(
    unseen_items[
        [
            "item_id",
            "title",
            "destination_norm",
            "destination_group",
            "category_mapped",
            "activity_type_mapped",
            "difficulty_level_norm",
            "duration_type_mapped",
        ]
    ]
)


Unseen canonical shape: (8, 38)
Unseen sparse feature matrix shape: (8, 10775)
Unseen dense vector shape: (8, 128)
Single-item dense vector length: 128


/opt/anaconda3/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['birds', 'cuisine', 'extreme adventure', 'helicopter', 'jungle', 'mountain view', 'sunrise flight', 'tandem', 'unesco'] will be ignored
  warnings.warn(
/opt/anaconda3/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['adventure seekers', 'couples', 'first-time visitors', 'groups', 'luxury travelers', 'nature-focused travelers', 'solo travelers', 'wellness travelers'] will be ignored
  warnings.warn(
/opt/anaconda3/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['birds', 'cuisine', 'extreme adventure', 'helicopter', 'jungle', 'mountain view', 'sunrise flight', 'tandem', 'unesco'] will be ignored
  warnings.warn(
/opt/anaconda3/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['adventure seekers', 'couples', 'first-time visitors', '

,item_id,title,destination_norm,destination_group,category_mapped,activity_type_mapped,difficulty_level_norm,duration_type_mapped
0,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,Kathmandu UNESCO Heritage Walk with Local Storytelling Guide,kathmandu,kathmandu_valley,heritage,city_tour,easy,full_day
1,unseen_pokhara_lakeside_sunrise_paragliding_exp_666e709e124e1ac5,Pokhara Lakeside Sunrise Paragliding Experience,pokhara,pokhara_annapurna,adventure,paragliding,easy,half_day
2,unseen_chitwan_jeep_and_wildlife_observation_sa_e21c3b8cd7c0f279,Chitwan Jeep and Wildlife Observation Safari,chitwan,chitwan_region,nature,wildlife_safari,easy,full_day
3,unseen_bhaktapur_pottery_and_old_town_cultural__6f63e8452c50be56,Bhaktapur Pottery and Old Town Cultural Workshop,bhaktapur,kathmandu_valley,heritage,cultural_workshop,easy,half_day
4,unseen_nagarkot_morning_yoga_with_himalayan_vie_faa740c67a7afed7,Nagarkot Morning Yoga with Himalayan Viewpoint Session,nagarkot,kathmandu_valley,wellness,yoga,easy,half_day
5,unseen_bhote_koshi_extreme_bungee_adventure_day_59aea562f73058b8,Bhote Koshi Extreme Bungee Adventure Day,bhote koshi,bhote_koshi_region,adventure,bungee,moderate,full_day
6,unseen_pokhara_nepali_cooking_class_with_market_4e89929be335a2b7,Pokhara Nepali Cooking Class with Market Visit,pokhara,pokhara_annapurna,food_tours,cooking_class,easy,half_day
7,unseen_everest_region_scenic_helicopter_mountai_2017795ce2da5429,Everest Region Scenic Helicopter Mountain Flight,everest region,everest_region,adventure,mountain_flight,easy,half_day


In [21]:
unseen_seed_position = 0

print("UNSEEN SEED:")
display(
    unseen_items.iloc[
        [unseen_seed_position]
    ][
        [
            "item_id",
            "title",
            "destination",
            "category_mapped",
            "activity_type_mapped",
            "duration_type_mapped",
        ]
    ]
)

print("\nSPARSE OFFLINE DEBUG PATH")
display(
    recommend_from_matrix(
        query_items=unseen_items,
        query_matrix=X_unseen_sparse,
        query_row_position=unseen_seed_position,
        candidate_frame=train_items,
        knn_model=sparse_knn,
        feature_config=best_feature_config,
        retrieval_path="sparse_offline",
        top_k=best_feature_config["top_k"],
        exclude_self=False,
    )
)

print("\nDENSE PRODUCTION BRIDGE PATH")
display(
    recommend_from_matrix(
        query_items=unseen_items,
        query_matrix=X_unseen_dense,
        query_row_position=unseen_seed_position,
        candidate_frame=train_items,
        knn_model=dense_knn,
        feature_config=best_feature_config,
        retrieval_path="dense_pgvector_bridge",
        top_k=best_feature_config["top_k"],
        exclude_self=False,
    )
)


UNSEEN SEED:


,item_id,title,destination,category_mapped,activity_type_mapped,duration_type_mapped
0,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,Kathmandu UNESCO Heritage Walk with Local Storytelling Guide,Kathmandu,heritage,city_tour,full_day



SPARSE OFFLINE DEBUG PATH


,query_item_id,recommended_item_id,title,destination,destination_norm,destination_group,category,category_mapped,activity_type,activity_type_mapped,difficulty_level,duration_text,cosine_similarity,reranked_score,candidate_row_position,retrieval_path
0,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,Kathmandu UNESCO Heritage Circuit with Private Guide,Kathmandu,kathmandu,kathmandu_valley,Culture | Landmarks,heritage,CITY_TOUR,city_tour,EASY,7 hours,0.725146,1.218053,0,sparse_offline
1,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,scraped_kathmandu_seven_unesco_heritage_sites_to_ccc5c90b91651ec4,Kathmandu Seven UNESCO Heritage Sites Tour,Kathmandu,kathmandu,kathmandu_valley,Culture | Landmarks,heritage,CITY_TOUR,city_tour,EASY,9 hours,0.716085,1.208992,18,sparse_offline
2,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,scraped_pashupatinath_boudhanath_and_patan_herit_c1faa00f8590b3d8,"Pashupatinath, Boudhanath and Patan Heritage Day",Kathmandu,kathmandu,kathmandu_valley,Old towns | Guided touring,heritage,CITY_TOUR,city_tour,EASY,5.5 hours,0.690304,1.183211,1,sparse_offline
3,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,scraped_chobar_gorge_kirtipur_and_taudaha_half_d_c602a12f2c95862b,"Chobar Gorge, Kirtipur and Taudaha Half Day",Kathmandu,kathmandu,kathmandu_valley,Heritage | Local neighborhoods,heritage,CITY_TOUR,city_tour,EASY,8 hours,0.633375,1.126281,1091,sparse_offline
4,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,scraped_kathmandu_bahal_courtyards_and_hidden_sh_b7b71cce1fa29638,Kathmandu Bahal Courtyards and Hidden Shrines Walk,Kathmandu,kathmandu,kathmandu_valley,Urban heritage | Walking routes,heritage,CITY_TOUR,city_tour,EASY,3.5 hours,0.618443,1.111349,365,sparse_offline



DENSE PRODUCTION BRIDGE PATH


,query_item_id,recommended_item_id,title,destination,destination_norm,destination_group,category,category_mapped,activity_type,activity_type_mapped,difficulty_level,duration_text,cosine_similarity,reranked_score,candidate_row_position,retrieval_path
0,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,Kathmandu UNESCO Heritage Circuit with Private Guide,Kathmandu,kathmandu,kathmandu_valley,Culture | Landmarks,heritage,CITY_TOUR,city_tour,EASY,7 hours,0.996081,1.488987,0,dense_pgvector_bridge
1,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,scraped_kathmandu_seven_unesco_heritage_sites_to_ccc5c90b91651ec4,Kathmandu Seven UNESCO Heritage Sites Tour,Kathmandu,kathmandu,kathmandu_valley,Culture | Landmarks,heritage,CITY_TOUR,city_tour,EASY,9 hours,0.978889,1.471795,18,dense_pgvector_bridge
2,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,scraped_pashupatinath_boudhanath_and_patan_herit_c1faa00f8590b3d8,"Pashupatinath, Boudhanath and Patan Heritage Day",Kathmandu,kathmandu,kathmandu_valley,Old towns | Guided touring,heritage,CITY_TOUR,city_tour,EASY,5.5 hours,0.977797,1.470704,1,dense_pgvector_bridge
3,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,scraped_chobar_gorge_kirtipur_and_taudaha_half_d_c602a12f2c95862b,"Chobar Gorge, Kirtipur and Taudaha Half Day",Kathmandu,kathmandu,kathmandu_valley,Heritage | Local neighborhoods,heritage,CITY_TOUR,city_tour,EASY,8 hours,0.876178,1.369084,1091,dense_pgvector_bridge
4,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,scraped_bhaktapur_and_patan_highlights_from_kath_34eb585502c955e3,Bhaktapur and Patan Highlights from Kathmandu,Kathmandu Valley,kathmandu valley,kathmandu_valley,Old towns | Guided touring,heritage,CITY_TOUR,city_tour,EASY,8.5 hours,0.869249,1.352346,9,dense_pgvector_bridge


In [22]:
for row_position in range(len(unseen_items)):
    seed_row = unseen_items.iloc[row_position]
    print("=" * 120)
    print("UNSEEN SEED:", seed_row["title"], "|", seed_row["item_id"])
    display(
        unseen_items.iloc[
            [row_position]
        ][
            [
                "destination",
                "category_mapped",
                "activity_type_mapped",
                "difficulty_level_norm",
                "duration_type_mapped",
            ]
        ]
    )
    print("Sparse offline debug recommendations")
    display(
        recommend_from_matrix(
            query_items=unseen_items,
            query_matrix=X_unseen_sparse,
            query_row_position=row_position,
            candidate_frame=train_items,
            knn_model=sparse_knn,
            feature_config=best_feature_config,
            retrieval_path="sparse_offline",
            top_k=best_feature_config["top_k"],
            exclude_self=False,
        )
    )
    print("Dense production bridge recommendations")
    display(
        recommend_from_matrix(
            query_items=unseen_items,
            query_matrix=X_unseen_dense,
            query_row_position=row_position,
            candidate_frame=train_items,
            knn_model=dense_knn,
            feature_config=best_feature_config,
            retrieval_path="dense_pgvector_bridge",
            top_k=best_feature_config["top_k"],
            exclude_self=False,
        )
    )


UNSEEN SEED: Kathmandu UNESCO Heritage Walk with Local Storytelling Guide | unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a


,destination,category_mapped,activity_type_mapped,difficulty_level_norm,duration_type_mapped
0,Kathmandu,heritage,city_tour,easy,full_day


Sparse offline debug recommendations


,query_item_id,recommended_item_id,title,destination,destination_norm,destination_group,category,category_mapped,activity_type,activity_type_mapped,difficulty_level,duration_text,cosine_similarity,reranked_score,candidate_row_position,retrieval_path
0,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,Kathmandu UNESCO Heritage Circuit with Private Guide,Kathmandu,kathmandu,kathmandu_valley,Culture | Landmarks,heritage,CITY_TOUR,city_tour,EASY,7 hours,0.725146,1.218053,0,sparse_offline
1,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,scraped_kathmandu_seven_unesco_heritage_sites_to_ccc5c90b91651ec4,Kathmandu Seven UNESCO Heritage Sites Tour,Kathmandu,kathmandu,kathmandu_valley,Culture | Landmarks,heritage,CITY_TOUR,city_tour,EASY,9 hours,0.716085,1.208992,18,sparse_offline
2,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,scraped_pashupatinath_boudhanath_and_patan_herit_c1faa00f8590b3d8,"Pashupatinath, Boudhanath and Patan Heritage Day",Kathmandu,kathmandu,kathmandu_valley,Old towns | Guided touring,heritage,CITY_TOUR,city_tour,EASY,5.5 hours,0.690304,1.183211,1,sparse_offline
3,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,scraped_chobar_gorge_kirtipur_and_taudaha_half_d_c602a12f2c95862b,"Chobar Gorge, Kirtipur and Taudaha Half Day",Kathmandu,kathmandu,kathmandu_valley,Heritage | Local neighborhoods,heritage,CITY_TOUR,city_tour,EASY,8 hours,0.633375,1.126281,1091,sparse_offline
4,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,scraped_kathmandu_bahal_courtyards_and_hidden_sh_b7b71cce1fa29638,Kathmandu Bahal Courtyards and Hidden Shrines Walk,Kathmandu,kathmandu,kathmandu_valley,Urban heritage | Walking routes,heritage,CITY_TOUR,city_tour,EASY,3.5 hours,0.618443,1.111349,365,sparse_offline


Dense production bridge recommendations


,query_item_id,recommended_item_id,title,destination,destination_norm,destination_group,category,category_mapped,activity_type,activity_type_mapped,difficulty_level,duration_text,cosine_similarity,reranked_score,candidate_row_position,retrieval_path
0,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,scraped_kathmandu_unesco_heritage_circuit_with_p_d5e548f8d7da4f8c,Kathmandu UNESCO Heritage Circuit with Private Guide,Kathmandu,kathmandu,kathmandu_valley,Culture | Landmarks,heritage,CITY_TOUR,city_tour,EASY,7 hours,0.996081,1.488987,0,dense_pgvector_bridge
1,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,scraped_kathmandu_seven_unesco_heritage_sites_to_ccc5c90b91651ec4,Kathmandu Seven UNESCO Heritage Sites Tour,Kathmandu,kathmandu,kathmandu_valley,Culture | Landmarks,heritage,CITY_TOUR,city_tour,EASY,9 hours,0.978889,1.471795,18,dense_pgvector_bridge
2,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,scraped_pashupatinath_boudhanath_and_patan_herit_c1faa00f8590b3d8,"Pashupatinath, Boudhanath and Patan Heritage Day",Kathmandu,kathmandu,kathmandu_valley,Old towns | Guided touring,heritage,CITY_TOUR,city_tour,EASY,5.5 hours,0.977797,1.470704,1,dense_pgvector_bridge
3,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,scraped_chobar_gorge_kirtipur_and_taudaha_half_d_c602a12f2c95862b,"Chobar Gorge, Kirtipur and Taudaha Half Day",Kathmandu,kathmandu,kathmandu_valley,Heritage | Local neighborhoods,heritage,CITY_TOUR,city_tour,EASY,8 hours,0.876178,1.369084,1091,dense_pgvector_bridge
4,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,scraped_bhaktapur_and_patan_highlights_from_kath_34eb585502c955e3,Bhaktapur and Patan Highlights from Kathmandu,Kathmandu Valley,kathmandu valley,kathmandu_valley,Old towns | Guided touring,heritage,CITY_TOUR,city_tour,EASY,8.5 hours,0.869249,1.352346,9,dense_pgvector_bridge


... (38 outputs left out)

Sparse offline debug recommendations


,query_item_id,recommended_item_id,title,destination,destination_norm,destination_group,category,category_mapped,activity_type,activity_type_mapped,difficulty_level,duration_text,cosine_similarity,reranked_score,candidate_row_position,retrieval_path
0,unseen_everest_region_scenic_helicopter_mountai_2017795ce2da5429,scraped_everest_view_helicopter_flight_from_kath_cb227ae0a917f73c,Everest View Helicopter Flight from Kathmandu,Everest Region,everest region,everest_region,Air tours | Signature experiences,nature,MOUNTAIN_FLIGHT,mountain_flight,EASY,3 hours,0.677549,1.047473,178,sparse_offline
1,unseen_everest_region_scenic_helicopter_mountai_2017795ce2da5429,scraped_everest_panorama_helicopter_stopover_bre_c3b48498d0a64ba3,Everest Panorama Helicopter Stopover Breakfast,Kathmandu,kathmandu,kathmandu_valley,Air tours | Helicopter outings,adventure,MOUNTAIN_FLIGHT,mountain_flight,EASY,4 hours,0.654288,0.982446,807,sparse_offline
2,unseen_everest_region_scenic_helicopter_mountai_2017795ce2da5429,scraped_pokhara_ultralight_scenic_flight_over_ph_f7ba9d99ea023e14,Pokhara Ultralight Scenic Flight over Phewa Valley,Pokhara,pokhara,pokhara_annapurna,Air tours | Scenic flights,adventure,MOUNTAIN_FLIGHT,mountain_flight,EASY,1 hours,0.646896,0.975054,805,sparse_offline
3,unseen_everest_region_scenic_helicopter_mountai_2017795ce2da5429,scraped_early_morning_everest_scenic_flight_with_c271188347af77d4,Early Morning Everest Scenic Flight with Transfers,Kathmandu,kathmandu,kathmandu_valley,Air tours | Everest views,adventure,MOUNTAIN_FLIGHT,mountain_flight,EASY,3 hours,0.640090,0.968248,76,sparse_offline
4,unseen_everest_region_scenic_helicopter_mountai_2017795ce2da5429,scraped_everest_panorama_mountain_flight_with_ai_129a782a767cc64a,Everest Panorama Mountain Flight with Airport Transfers,Kathmandu,kathmandu,kathmandu_valley,Flightseeing | Himalayan views,adventure,MOUNTAIN_FLIGHT,mountain_flight,EASY,3 hours,0.610563,0.938721,72,sparse_offline


Dense production bridge recommendations


,query_item_id,recommended_item_id,title,destination,destination_norm,destination_group,category,category_mapped,activity_type,activity_type_mapped,difficulty_level,duration_text,cosine_similarity,reranked_score,candidate_row_position,retrieval_path
0,unseen_everest_region_scenic_helicopter_mountai_2017795ce2da5429,scraped_everest_view_helicopter_flight_from_kath_cb227ae0a917f73c,Everest View Helicopter Flight from Kathmandu,Everest Region,everest region,everest_region,Air tours | Signature experiences,nature,MOUNTAIN_FLIGHT,mountain_flight,EASY,3 hours,0.819620,1.189545,178,dense_pgvector_bridge
1,unseen_everest_region_scenic_helicopter_mountai_2017795ce2da5429,scraped_early_morning_everest_scenic_flight_with_c271188347af77d4,Early Morning Everest Scenic Flight with Transfers,Kathmandu,kathmandu,kathmandu_valley,Air tours | Everest views,adventure,MOUNTAIN_FLIGHT,mountain_flight,EASY,3 hours,0.815095,1.143252,76,dense_pgvector_bridge
2,unseen_everest_region_scenic_helicopter_mountai_2017795ce2da5429,scraped_everest_panorama_mountain_flight_with_ai_129a782a767cc64a,Everest Panorama Mountain Flight with Airport Transfers,Kathmandu,kathmandu,kathmandu_valley,Flightseeing | Himalayan views,adventure,MOUNTAIN_FLIGHT,mountain_flight,EASY,3 hours,0.814205,1.142362,72,dense_pgvector_bridge
3,unseen_everest_region_scenic_helicopter_mountai_2017795ce2da5429,scraped_kathmandu_everest_mountain_flight_experi_b75eede0323378e0,Kathmandu Everest Mountain Flight Experience,Kathmandu,kathmandu,kathmandu_valley,Scenic flights | Mountain panoramas,adventure,MOUNTAIN_FLIGHT,mountain_flight,EASY,3 hours,0.810141,1.138298,74,dense_pgvector_bridge
4,unseen_everest_region_scenic_helicopter_mountai_2017795ce2da5429,scraped_everest_scenic_flight_with_hotel_transfe_8e3172b1e63b40b9,Everest Scenic Flight with Hotel Transfers,Kathmandu,kathmandu,kathmandu_valley,Scenic flights | Mountain panoramas,adventure,MOUNTAIN_FLIGHT,mountain_flight,EASY,3 hours,0.809041,1.137199,75,dense_pgvector_bridge


In [23]:
unseen_comparison_df, unseen_comparison_summary, unseen_sparse_cache, unseen_dense_cache = compare_query_sets_against_training(
    query_items=unseen_items,
    sparse_query_matrix=X_unseen_sparse,
    dense_query_matrix=X_unseen_dense,
    candidate_items=train_items,
    sparse_knn=sparse_knn,
    dense_knn=dense_knn,
    feature_config=best_feature_config,
    top_k=best_feature_config["top_k"],
)

unseen_summary_display = pd.DataFrame(
    [
        {"path": "sparse_offline", **unseen_comparison_summary["sparse"]},
        {"path": "dense_pgvector_bridge", **unseen_comparison_summary["dense"]},
        {"path": "dense_vs_sparse", **unseen_comparison_summary["dense_vs_sparse"]},
    ]
)

print("Unseen query comparison summary:")
display(unseen_summary_display)
display(unseen_comparison_df)


Unseen query comparison summary:


,path,n_queries,category_match_rate_at_k_mean,activity_type_match_rate_at_k_mean,destination_match_rate_at_k_mean,destination_group_match_rate_at_k_mean,avg_cosine_similarity_at_k_mean,overlap_at_k_mean,category_match_delta_mean,activity_type_match_delta_mean,destination_match_delta_mean,destination_group_match_delta_mean,avg_cosine_similarity_delta_mean
0,sparse_offline,8.0,0.95,1.0,0.700,0.775,0.682327,NaN,NaN,NaN,NaN,NaN,NaN
1,dense_pgvector_bridge,8.0,0.95,1.0,0.675,0.775,0.896478,NaN,NaN,NaN,NaN,NaN,NaN
2,dense_vs_sparse,NaN,NaN,NaN,NaN,NaN,NaN,0.725,0.0,0.0,-0.025,0.0,0.21415


,query_row_position,query_item_id,query_title,category_match_rate_at_k_sparse,activity_type_match_rate_at_k_sparse,destination_match_rate_at_k_sparse,destination_group_match_rate_at_k_sparse,avg_cosine_similarity_at_k_sparse,category_match_rate_at_k_dense,activity_type_match_rate_at_k_dense,destination_match_rate_at_k_dense,destination_group_match_rate_at_k_dense,avg_cosine_similarity_at_k_dense,overlap_at_k
0,0,unseen_kathmandu_unesco_heritage_walk_with_loca_61e28d30884faf0a,Kathmandu UNESCO Heritage Walk with Local Storytelling Guide,1.0,1.0,1.0,1.0,0.676671,1.0,1.0,0.8,1.0,0.939639,0.8
1,1,unseen_pokhara_lakeside_sunrise_paragliding_exp_666e709e124e1ac5,Pokhara Lakeside Sunrise Paragliding Experience,0.8,1.0,1.0,1.0,0.708661,0.8,1.0,1.0,1.0,0.886425,1.0
2,2,unseen_chitwan_jeep_and_wildlife_observation_sa_e21c3b8cd7c0f279,Chitwan Jeep and Wildlife Observation Safari,1.0,1.0,1.0,1.0,0.687055,1.0,1.0,1.0,1.0,0.858977,0.6
3,3,unseen_bhaktapur_pottery_and_old_town_cultural__6f63e8452c50be56,Bhaktapur Pottery and Old Town Cultural Workshop,1.0,1.0,1.0,1.0,0.772668,1.0,1.0,1.0,1.0,0.967735,1.0
4,4,unseen_nagarkot_morning_yoga_with_himalayan_vie_faa740c67a7afed7,Nagarkot Morning Yoga with Himalayan Viewpoint Session,1.0,1.0,0.4,1.0,0.635351,1.0,1.0,0.4,1.0,0.824628,0.8
5,5,unseen_bhote_koshi_extreme_bungee_adventure_day_59aea562f73058b8,Bhote Koshi Extreme Bungee Adventure Day,1.0,1.0,0.0,0.0,0.603919,1.0,1.0,0.0,0.0,0.906523,0.8
6,6,unseen_pokhara_nepali_cooking_class_with_market_4e89929be335a2b7,Pokhara Nepali Cooking Class with Market Visit,1.0,1.0,1.0,1.0,0.728417,1.0,1.0,1.0,1.0,0.974276,0.2
7,7,unseen_everest_region_scenic_helicopter_mountai_2017795ce2da5429,Everest Region Scenic Helicopter Mountain Flight,0.8,1.0,0.2,0.2,0.645877,0.8,1.0,0.2,0.2,0.813620,0.6


# Export Artifacts

The production export below is transform-focused rather than recommender-class-focused.

Backend serving contract:
- Django should use the exported bundle to preprocess listings and build dense vectors.
- `pgvector` should perform nearest-neighbor retrieval over live DB listing vectors.
- Django should then apply business-rule filtering, caching, reranking, and API formatting.
- The notebook artifacts are not the final recommendation-serving engine that returns listing IDs directly.

Identity contract:
- Notebook `item_id` values are training and evaluation identifiers only.
- Django live indexing must use the real DB primary key or UUID as the production item identity.

In [24]:
created_at_utc = datetime.now(timezone.utc).isoformat()

artifact_filenames = {
    "production_transform_bundle": "production_transform_bundle.pkl",
    "artifact_manifest": "artifact_manifest.json",
    "preprocessing_contract": "preprocessing_contract.json",
    "feature_config": "feature_config.json",
    "mapping_config": "mapping_config.json",
    "training_metrics_summary": "training_metrics_summary.json",
    "canonical_scraped_final": "canonical_scraped_final.csv",
    "offline_recommender_debug": "offline_recommender_debug.pkl",
    "evaluation_metrics_per_item": "evaluation_metrics_per_item.csv",
    "sample_recommendations": "sample_recommendations.md",
    "transform_sanity_samples": "transform_sanity_samples.json",
}

artifact_paths = {
    artifact_name: ARTIFACT_DIR / filename
    for artifact_name, filename in artifact_filenames.items()
}

category_mapping_contract = {
    "rule_order": CATEGORY_RULE_SEQUENCE,
    "activity_to_category_rules": CATEGORY_ACTIVITY_RULES,
    "keyword_rules": CATEGORY_KEYWORD_RULES,
    "text_fields_used": CATEGORY_TEXT_FIELDS_USED,
    "fallback_category": CATEGORY_FALLBACK,
    "implementation_note": "Backend Python should preserve this exact rule order and OR-semantics between activity matches and keyword matches.",
}

exported_feature_config = {
    "artifact_version": ARTIFACT_VERSION,
    "schema_version": SCHEMA_VERSION,
    "text_base_columns": TEXT_BASE_COLS,
    "categorical_columns": CATEGORICAL_COLS,
    "multi_value_columns": MULTI_VALUE_COLS,
    "numeric_columns": NUMERIC_COLS,
    "semantic_boolean_columns": SEMANTIC_BOOLEAN_COLS,
    "non_semantic_runtime_fields": NON_SEMANTIC_RUNTIME_FIELDS,
    "weights": best_feature_config["weights"],
    "rerank_bonus": best_feature_config["rerank_bonus"],
    "candidate_pool": best_feature_config["candidate_pool"],
    "top_k": best_feature_config["top_k"],
    "tfidf": best_feature_config["tfidf"],
    "dense_projection": {
        **best_feature_config["dense_projection"],
        "actual_fitted_dim": int(X_train_dense.shape[1]),
        "explained_variance_ratio_sum": float(final_dense_projector.explained_variance_ratio_.sum()),
    },
    "prefixed_token_recipe": PREFIXED_TOKEN_RECIPE,
    "combined_text_recipe": {
        "title_repeat": 2,
        "destination_repeat": 2,
        "destination_group_repeat": 2,
        "meta_token_text_included": True,
    },
    "notes": [
        "Sparse TF-IDF plus metadata is retained for notebook-side evaluation only.",
        "Dense vectors are the production-facing pgvector bridge.",
        "NON_SEMANTIC_RUNTIME_FIELDS are excluded from the semantic vector space.",
        BACKEND_SERVING_NOTE,
    ],
}

mapping_config = {
    "destination_aliases": DESTINATION_ALIASES,
    "destination_group_map": DESTINATION_GROUP_MAP,
    "activity_map": ACTIVITY_MAP,
    "difficulty_fallback_map": DIFFICULTY_FALLBACK_MAP,
    "duration_type_map": DURATION_TYPE_MAP,
    "category_mapping_contract": category_mapping_contract,
}

preprocessing_contract = {
    "artifact_version": ARTIFACT_VERSION,
    "schema_version": SCHEMA_VERSION,
    "required_raw_input_fields": REQUIRED_RAW_COLUMNS,
    "canonical_output_fields": CANONICAL_OUTPUT_COLUMNS,
    "categorical_columns": CATEGORICAL_COLS,
    "multi_value_columns": MULTI_VALUE_COLS,
    "numeric_columns": NUMERIC_COLS,
    "boolean_columns": BOOLEAN_COLS,
    "semantic_boolean_columns": SEMANTIC_BOOLEAN_COLS,
    "non_semantic_runtime_fields": NON_SEMANTIC_RUNTIME_FIELDS,
    "default_fill_behavior": DEFAULT_FILL_BEHAVIOR,
    "expected_duration_type_values": EXPECTED_DURATION_TYPE_VALUES,
    "expected_difficulty_values": EXPECTED_DIFFICULTY_VALUES,
    "identity_contract": IDENTITY_CONTRACT,
    "notes": [
        "Use prepare_items() as the canonical logical contract for both training and live listings.",
        "split_pipe() accepts pipe-delimited strings, JSON-array strings, and Python iterable inputs.",
        "Half-day is preserved as half_day and is not collapsed into hourly.",
        "Category mapping must preserve the exported rule order and fallback behavior exactly.",
    ],
}

production_transform_bundle = {
    "artifact_version": ARTIFACT_VERSION,
    "schema_version": SCHEMA_VERSION,
    "created_at_utc": created_at_utc,
    "production_intent": "pgvector retrieval over live Django listing vectors",
    "required_raw_input_fields": REQUIRED_RAW_COLUMNS,
    "canonical_output_fields": CANONICAL_OUTPUT_COLUMNS,
    "text_base_columns": TEXT_BASE_COLS,
    "categorical_columns": CATEGORICAL_COLS,
    "multi_value_columns": MULTI_VALUE_COLS,
    "numeric_columns": NUMERIC_COLS,
    "semantic_boolean_columns": SEMANTIC_BOOLEAN_COLS,
    "non_semantic_runtime_fields": NON_SEMANTIC_RUNTIME_FIELDS,
    "identity_contract": IDENTITY_CONTRACT,
    "backend_serving_note": BACKEND_SERVING_NOTE,
    "feature_config": best_feature_config,
    "prefixed_token_recipe": PREFIXED_TOKEN_RECIPE,
    "mapping_config": mapping_config,
    "vectorizer": final_feature_components["vectorizer"],
    "categorical_encoder": final_feature_components["cat_encoder"],
    "multilabel_binarizers": final_feature_components["multilabel_binarizers"],
    "numeric_scaler": final_feature_components["scaler"],
    "dense_projector": final_dense_projector,
    "dense_vector_dimension": int(X_train_dense.shape[1]),
    "dense_vector_normalization": best_feature_config["dense_projection"]["normalization"],
}

offline_recommender_debug = {
    "artifact_version": ARTIFACT_VERSION,
    "created_at_utc": created_at_utc,
    "purpose": "Notebook-only sparse and dense KNN sanity checking.",
    "feature_config": best_feature_config,
    "train_items": train_items,
    "sparse_knn": sparse_knn,
    "dense_knn": dense_knn,
    "X_train_sparse": X_train_sparse,
    "X_train_dense": X_train_dense,
}

sanity_sample_count = min(3, len(train_items))
transform_sanity_samples = {
    "artifact_version": ARTIFACT_VERSION,
    "created_at_utc": created_at_utc,
    "dense_vector_dimension": int(X_train_dense.shape[1]),
    "train_samples": [
        {
            "item_id": train_items.iloc[row_position]["item_id"],
            "title": train_items.iloc[row_position]["title"],
            "destination_norm": train_items.iloc[row_position]["destination_norm"],
            "category_mapped": train_items.iloc[row_position]["category_mapped"],
            "activity_type_mapped": train_items.iloc[row_position]["activity_type_mapped"],
            "duration_type_mapped": train_items.iloc[row_position]["duration_type_mapped"],
            "dense_vector": X_train_dense[row_position].tolist(),
        }
        for row_position in range(sanity_sample_count)
    ],
    "single_item_transform_example": {
        "item_id": prepared_single_item.iloc[0]["item_id"],
        "title": prepared_single_item.iloc[0]["title"],
        "destination_norm": prepared_single_item.iloc[0]["destination_norm"],
        "category_mapped": prepared_single_item.iloc[0]["category_mapped"],
        "activity_type_mapped": prepared_single_item.iloc[0]["activity_type_mapped"],
        "duration_type_mapped": prepared_single_item.iloc[0]["duration_type_mapped"],
        "dense_vector": single_dense_vector.tolist(),
    },
}

training_metrics_summary = {
    "artifact_version": ARTIFACT_VERSION,
    "created_at_utc": created_at_utc,
    "schema_version": SCHEMA_VERSION,
    "training_dataset_size": int(len(train_items)),
    "vector_dimension": int(X_train_dense.shape[1]),
    "dense_explained_variance_ratio_sum": float(final_dense_projector.explained_variance_ratio_.sum()),
    "training_comparison": comparison_summary,
    "unseen_query_comparison": unseen_comparison_summary,
    "identity_contract": IDENTITY_CONTRACT,
    "backend_serving_note": BACKEND_SERVING_NOTE,
}

if "study" in globals() and hasattr(study, "best_params"):
    training_metrics_summary["optuna_best_params"] = make_json_safe(study.best_params)

artifact_manifest = {
    "artifact_version": ARTIFACT_VERSION,
    "created_at_utc": created_at_utc,
    "schema_version": SCHEMA_VERSION,
    "training_dataset_size": int(len(train_items)),
    "vector_dimension": int(X_train_dense.shape[1]),
    "sparse_feature_dimension": int(X_train_sparse.shape[1]),
    "production_intent": "pgvector retrieval",
    "identity_contract": IDENTITY_CONTRACT,
    "backend_serving_note": BACKEND_SERVING_NOTE,
    "artifact_filenames": artifact_filenames,
    "notes": [
        "The notebook is not the production serving engine.",
        "The production-facing export is the transform bundle for generating dense vectors.",
        "Django should use DB PK or UUID as live item identity, not notebook item_id values.",
    ],
}

display(
    pd.DataFrame(
        [
            {"artifact_name": artifact_name, "filename": filename}
            for artifact_name, filename in artifact_filenames.items()
        ]
    )
)

,artifact_name,filename
0,production_transform_bundle,production_transform_bundle.pkl
1,artifact_manifest,artifact_manifest.json
2,preprocessing_contract,preprocessing_contract.json
3,feature_config,feature_config.json
4,mapping_config,mapping_config.json
5,training_metrics_summary,training_metrics_summary.json
6,canonical_scraped_final,canonical_scraped_final.csv
7,offline_recommender_debug,offline_recommender_debug.pkl
8,evaluation_metrics_per_item,evaluation_metrics_per_item.csv
9,sample_recommendations,sample_recommendations.md


In [25]:
joblib.dump(production_transform_bundle, artifact_paths["production_transform_bundle"])
joblib.dump(offline_recommender_debug, artifact_paths["offline_recommender_debug"])

with open(artifact_paths["artifact_manifest"], "w", encoding="utf-8") as file_handle:
    json.dump(make_json_safe(artifact_manifest), file_handle, indent=2)

with open(artifact_paths["preprocessing_contract"], "w", encoding="utf-8") as file_handle:
    json.dump(make_json_safe(preprocessing_contract), file_handle, indent=2)

with open(artifact_paths["feature_config"], "w", encoding="utf-8") as file_handle:
    json.dump(make_json_safe(exported_feature_config), file_handle, indent=2)

with open(artifact_paths["mapping_config"], "w", encoding="utf-8") as file_handle:
    json.dump(make_json_safe(mapping_config), file_handle, indent=2)

with open(artifact_paths["training_metrics_summary"], "w", encoding="utf-8") as file_handle:
    json.dump(make_json_safe(training_metrics_summary), file_handle, indent=2)

with open(artifact_paths["transform_sanity_samples"], "w", encoding="utf-8") as file_handle:
    json.dump(make_json_safe(transform_sanity_samples), file_handle, indent=2)

train_items.to_csv(artifact_paths["canonical_scraped_final"], index=False)
comparison_metrics_df.to_csv(artifact_paths["evaluation_metrics_per_item"], index=False)

sample_row_positions = [row for row in [0, 100, 300, 700, 1000] if row < len(train_items)]
with open(artifact_paths["sample_recommendations"], "w", encoding="utf-8") as report_file:
    report_file.write("# Sample Recommendations\n\n")
    report_file.write("Sparse KNN is kept for notebook-side debug only. Dense vectors are the production pgvector bridge.\n\n")
    report_file.write("Transform-only note: the exported bundle is not the direct recommendation-serving engine.\n\n")

    for row_position in sample_row_positions:
        seed_row = train_items.iloc[row_position]
        sparse_recommendations = recommend_from_matrix(
            query_items=train_items,
            query_matrix=X_train_sparse,
            query_row_position=row_position,
            candidate_frame=train_items,
            knn_model=sparse_knn,
            feature_config=best_feature_config,
            retrieval_path="sparse_offline",
            top_k=best_feature_config["top_k"],
            exclude_self=True,
        )
        dense_recommendations = recommend_from_matrix(
            query_items=train_items,
            query_matrix=X_train_dense,
            query_row_position=row_position,
            candidate_frame=train_items,
            knn_model=dense_knn,
            feature_config=best_feature_config,
            retrieval_path="dense_pgvector_bridge",
            top_k=best_feature_config["top_k"],
            exclude_self=True,
        )

        report_file.write(f"## Seed: {seed_row['title']} ({seed_row['item_id']})\n\n")
        report_file.write("### Sparse Offline Debug Path\n\n")
        for _, row in sparse_recommendations.iterrows():
            report_file.write(
                f"- {row['recommended_item_id']} | {row['title']} | {row['destination']} | "
                f"{row['activity_type_mapped']} | cosine={row['cosine_similarity']:.4f} | "
                f"score={row['reranked_score']:.4f}\n"
            )

        report_file.write("\n### Dense Production Bridge Path\n\n")
        for _, row in dense_recommendations.iterrows():
            report_file.write(
                f"- {row['recommended_item_id']} | {row['title']} | {row['destination']} | "
                f"{row['activity_type_mapped']} | cosine={row['cosine_similarity']:.4f} | "
                f"score={row['reranked_score']:.4f}\n"
            )
        report_file.write("\n")

print("Saved artifacts:")
for artifact_name, artifact_path in artifact_paths.items():
    print(f"- {artifact_name}: {artifact_path}")

Saved artifacts:
- production_transform_bundle: tourism_recommender_artifacts/production_transform_bundle.pkl
- artifact_manifest: tourism_recommender_artifacts/artifact_manifest.json
- preprocessing_contract: tourism_recommender_artifacts/preprocessing_contract.json
- feature_config: tourism_recommender_artifacts/feature_config.json
- mapping_config: tourism_recommender_artifacts/mapping_config.json
- training_metrics_summary: tourism_recommender_artifacts/training_metrics_summary.json
- canonical_scraped_final: tourism_recommender_artifacts/canonical_scraped_final.csv
- offline_recommender_debug: tourism_recommender_artifacts/offline_recommender_debug.pkl
- evaluation_metrics_per_item: tourism_recommender_artifacts/evaluation_metrics_per_item.csv
- sample_recommendations: tourism_recommender_artifacts/sample_recommendations.md
- transform_sanity_samples: tourism_recommender_artifacts/transform_sanity_samples.json


# Summary Of Changes

- `prepare_items()` is now the single canonical preprocessing entry point for both scraped training rows and unseen or live-style rows.
- `split_pipe()` now supports pipe-delimited strings, JSON-array strings, and Python iterable inputs so the preprocessing contract is not scraped-data-specific.
- Notebook `item_id` is a training and evaluation identifier for notebook/export artifacts only. Django live indexing must use the real DB primary key or UUID as the production identity.
- The sparse TF-IDF + metadata KNN path is retained only for offline debugging, evaluation, and qualitative sanity checks.
- The dense `TruncatedSVD` + L2-normalized path is the production-facing bridge for later Django + `pgvector` retrieval over live DB listing vectors.
- The exported bundle is for transform only, not direct recommendation serving.

Artifact purposes:
- `production_transform_bundle.pkl`: fitted sklearn components and metadata for transforming live listings into dense vectors.
- `artifact_manifest.json`: artifact inventory, versioning, training size, vector dimension, identity contract, and production intent.
- `preprocessing_contract.json`: raw-input and canonical-output schema contract for Django preprocessing parity.
- `feature_config.json`: feature weights, TF-IDF settings, dense projection settings, and the full prefixed-token recipe.
- `mapping_config.json`: destination, activity, difficulty, duration, and category-mapping contracts.
- `training_metrics_summary.json`: sparse-vs-dense evaluation summary so the pgvector bridge can be sanity-checked.
- `canonical_scraped_final.csv`: canonical cleaned training dataframe, including notebook `item_id`.
- `offline_recommender_debug.pkl`: optional notebook-only sparse and dense KNN bundle for debugging.
- `evaluation_metrics_per_item.csv`: per-seed sparse-vs-dense evaluation metrics.
- `sample_recommendations.md`: human-readable recommendation snapshots for manual inspection.
- `transform_sanity_samples.json`: canonical sample rows plus dense vectors for backend transform parity checks.